<!-- dd:dd-lesson-np-3 -->

# Vectorization and broadcasting

*Numpy · `np-3`*

Work through this with the **Delta Drills** side panel open. It picks what you practise, sends you to the cell, and records how it went — you do not need to read this notebook in order.


In [ ]:
# === Delta Drills ===
# Which lesson this notebook is, for the side panel. Nothing to run.
DD_LESSON_ID = "np-3"


In [ ]:
#@title 🔧 Delta Drills checker — run me first { display-mode: "form" }
# Delta Drills — problem checker. Generated; see scripts/colab_grader.py.
#
# `dd_check(<problem id>)` runs your `solve` against the same cases the tutor
# grades with, and tells you which ones failed. It reads `solve` out of the
# notebook, so define it (run your cell) before you check.
import base64
import json
import sys
import zlib

import numpy as np

# Filled in by the generated cell that follows this source: {qid: {fn, cases}}.
_DD_TESTS = {}

# Where the ARENA digits fixture is fetched from, also filled in by that cell.
_DD_FIXTURE_URL = ""
_DD_FIXTURE_PATH = "/delta_numbers.npy"

_DD_RTOL = 1e-5
_DD_ATOL = 1e-6


def _dd_install_fixtures():
    """Make `np.load('/delta_numbers.npy')` work here the way it does in the app.

    24 of the einops drills are written against the ARENA digits image, and the
    bank refers to it by an absolute path the backend rewrites at grade time
    (`code_runner.CODE_PREAMBLE`). Nothing rewrote it in a notebook, so those
    problems could not run at all in Colab — not the checker, not the starter
    code the learner was sent there to fill in. Downloaded on first use, so the
    six notebooks that never touch it never pay for it.
    """
    import os
    import urllib.request

    original = np.load
    if getattr(original, "_dd_patched", False):
        return

    def _load(file, *args, **kwargs):
        if str(file) == _DD_FIXTURE_PATH and not os.path.exists(_DD_FIXTURE_PATH):
            if not _DD_FIXTURE_URL:
                raise FileNotFoundError(
                    "This drill needs the ARENA digits fixture and no source was "
                    "compiled into this notebook — regenerate it."
                )
            urllib.request.urlretrieve(_DD_FIXTURE_URL, _DD_FIXTURE_PATH)
        return original(file, *args, **kwargs)

    _load._dd_patched = True
    np.load = _load


def _dd_load(blob):
    """The test payload, deflated and base64'd.

    Not encryption and not pretending to be — it is one `zlib.decompress` away.
    It is compressed because the payload for a 84-problem notebook is ~80 KB of
    JSON, and out of sight because an expanded grader cell would otherwise sit
    in the notebook spelling out the expected answer to every problem below it.
    """
    return json.loads(zlib.decompress(base64.b64decode(blob)).decode("utf-8"))


def _dd_preflight_torch():
    """Import torch once, here, where a failure can still be explained.

    Every drill cell opens with `import torch as t`, so the learner meets a
    broken torch install as a traceback through torch's own internals — the one
    reported was `AttributeError: partially initialized module 'torch' has no
    attribute 'fx'` from `torch/_export/utils.py`, raised while evaluating a
    function's annotations. That message names neither the cause nor the cure,
    and it is not even the real error: it is what a LATER import sees after an
    earlier one died partway and left the half-built module in `sys.modules`.
    Python does unwind a failed import normally, but a torch that was swapped
    on disk under a running kernel (a `pip install` mid-session) or shadowed by
    a stray `torch.py` gets far enough in to be cached before it falls over.

    So: purge the wreckage and retry ONCE, which is the whole fix whenever the
    first failure was transient, and report what actually broke when it is not.
    Importing torch in this cell rather than lazily is safe now in a way the
    `_dd_tensor` comment below still guards against for the per-comparison
    path — the bank is 448/448 torch and every notebook imports it a few cells
    down, so there is no numpy-only notebook left to charge for it.

    Never raises: a checker that refuses to load over this would take the
    lesson down with the runtime.
    """

    def _purge():
        # Submodules too, and that is the whole point. Python drops only the
        # module that raised, so `torch` goes and a `torch._export` imported
        # seconds earlier STAYS — and the next `import torch` re-runs
        # `torch/__init__.py` straight back into that stale submodule, which
        # reaches for a `torch.fx` the half-built parent has not bound yet.
        # Leaving one behind reproduces the bug instead of clearing it.
        for name in [n for n in sys.modules if n == "torch" or n.startswith("torch.")]:
            del sys.modules[name]

    def _usable(mod):
        # `import torch` does NOT re-execute a module already in sys.modules,
        # so a corpse left by a failed import is imported "successfully" and
        # the error surfaces later, from the learner's own cell. Judge the
        # object, not the statement: a torch that finished has both of these.
        return hasattr(mod, "fx") and hasattr(mod, "__version__")

    cached = sys.modules.get("torch")
    if cached is not None and not _usable(cached):
        _purge()

    for attempt in (1, 2):
        try:
            import torch
            if not _usable(torch):
                raise ImportError(
                    "torch imported but is only partially initialised "
                    "(no .fx) — an earlier import in this session died partway"
                )
            return True
        except Exception as exc:
            if attempt == 1:
                _purge()
                continue
            print(
                "⚠️  This runtime cannot import PyTorch, so no drill in this "
                "notebook will run.\n"
                "    %s: %s\n"
                "    Fix: Runtime ▸ Disconnect and delete runtime, then reopen "
                "this notebook and run\n"
                "    this cell first. If it comes back, check for a file named "
                "torch.py in /content,\n"
                "    and re-run any pip install BEFORE anything imports torch."
                % (type(exc).__name__, exc)
            )
    return False


def _dd_tensor(value):
    # torch only if something already imported it. numpy-only notebooks must
    # not pay a torch import to compare two lists of ints.
    torch = sys.modules.get("torch")
    return torch is not None and isinstance(value, torch.Tensor)


def _dd_close(a, b):
    """Tolerance compare, but ONLY when a float or complex is involved.

    torch defaults to float32 where numpy defaults to float64 and honest
    answers differ in reduction order, so exact equality fails correct work.
    Integer and boolean results stay exact — an index answer (argmax, nonzero,
    searchsorted) must never be fudged by a tolerance. Returns None to mean
    "not a float comparison, use exact equality".
    """
    try:
        floaty = any(
            np.issubdtype(x.dtype, np.floating) or np.issubdtype(x.dtype, np.complexfloating)
            for x in (a, b)
        )
        if not floaty:
            return None
        if a.shape != b.shape:
            return False
        return bool(np.allclose(a, b, rtol=_DD_RTOL, atol=_DD_ATOL, equal_nan=True))
    except Exception:
        return None


def _dd_array_equal(a, b):
    close = _dd_close(a, b)
    if close is not None:
        return close
    return bool(np.array_equal(a, b))


def _dd_equal(a, b):
    if _dd_tensor(a) or _dd_tensor(b):
        try:
            a2 = a.detach().cpu().numpy() if _dd_tensor(a) else np.asarray(a)
            b2 = b.detach().cpu().numpy() if _dd_tensor(b) else np.asarray(b)
            return _dd_array_equal(a2, b2)
        except Exception:
            # dtypes numpy cannot hold (bfloat16, conj views): equal tensors
            # must not grade as unequal — ask torch itself.
            torch = sys.modules.get("torch")
            if torch is not None and isinstance(a, torch.Tensor) and isinstance(b, torch.Tensor):
                try:
                    return bool(torch.equal(a.detach().cpu().resolve_conj(),
                                            b.detach().cpu().resolve_conj()))
                except Exception:
                    return False
            return False
    if isinstance(a, np.ndarray) or isinstance(b, np.ndarray):
        return _dd_array_equal(np.asarray(a), np.asarray(b))
    if isinstance(a, (list, tuple)) and isinstance(b, (list, tuple)):
        if len(a) != len(b):
            return False
        return all(_dd_equal(x, y) for x, y in zip(a, b))
    close = _dd_close(np.asarray(a), np.asarray(b))
    if close is not None:
        return close
    return bool(a == b)


def _dd_seed():
    # The same seed before the actual-side and the expected-side setup runs, for
    # BOTH rngs: setup executes twice, so an unseeded torch.rand in a fixture
    # would hand the two sides different data and fail an honest answer.
    np.random.seed(0)
    torch = sys.modules.get("torch")
    if torch is not None:
        torch.manual_seed(0)


def _dd_show(value, limit=320):
    try:
        text = repr(value)
    except Exception as exc:
        text = "<unrepresentable: %s>" % type(exc).__name__
    text = " ".join(text.split()) if len(text) > limit else text
    if len(text) > limit:
        text = text[: limit - 1] + "…"
    return text


def dd_check(question_id, verbose=True):
    """Grade the `solve` you just defined against this problem's cases.

    Returns True when every case passes. Prints which ones did not, with the
    fixture, what was expected and what came back — a failing grade should be
    evidence you can act on, not a verdict.
    """
    qid = str(question_id)
    entry = _DD_TESTS.get(qid)
    if entry is None:
        print("No checker for problem %s in this notebook." % qid)
        return False

    # The learner's namespace, not this function's: `solve` lives in the cell
    # they ran, and in Colab that is the caller's globals.
    try:
        env = sys._getframe(1).f_globals
    except Exception:
        env = globals()

    fn_name = entry.get("fn") or "solve"
    if fn_name not in env:
        print("❌ `%s` is not defined yet — run your solution cell first." % fn_name)
        return False

    cases = entry.get("cases") or []
    failures = []
    for i, case in enumerate(cases, 1):
        # A fresh copy per case: fixtures are exec'd, and exec'ing them into the
        # notebook's own globals would quietly overwrite whatever the learner
        # named `x` two cells ago.
        ns = dict(env)
        try:
            if case.get("setup_code"):
                _dd_seed()
                exec(case["setup_code"], ns)
            actual = eval(case["call"], ns)
            expected_setup = case.get("expected_setup_code") or case.get("setup_code")
            if expected_setup:
                _dd_seed()
                exec(expected_setup, ns)
            expected = eval(case["expected_expr"], ns)
            if not _dd_equal(actual, expected):
                failures.append((i, case, _dd_show(expected), _dd_show(actual), ""))
        except Exception as exc:
            failures.append((i, case, "", "", "%s: %s" % (type(exc).__name__, exc)))

    total = len(cases)
    if not failures:
        print("✅ Problem %s — %d/%d cases passed." % (qid, total, total))
        return True

    print("❌ Problem %s — %d of %d cases failed." % (qid, len(failures), total))
    if verbose:
        for i, case, expected, actual, error in failures:
            print("\n  case %d" % i)
            if case.get("setup_code"):
                for line in case["setup_code"].strip().splitlines():
                    print("    given     %s" % line)
            print("    called    %s" % _dd_show_source(case.get("call", "")))
            if error:
                print("    raised    %s" % error)
            else:
                print("    expected  %s" % expected)
                print("    you got   %s" % actual)
    return False


def _dd_show_source(text, limit=160):
    text = " ".join(str(text).split())
    return text if len(text) <= limit else text[: limit - 1] + "…"

_DD_FIXTURE_URL = "https://raw.githubusercontent.com/AkiraTheSquid/arena-book-colab/main/ARENA_5.0/ch-1-foundations/numbers.npy"
_dd_preflight_torch()
_dd_install_fixtures()
_DD_TESTS = _dd_load(
    "eNrtPWlv20iyf4UI8DDSrKTX97HAAPvp/YL95hECJ1FmjHFkr+xkkiz2v7/qu3nIIpsU6eyuE8hyiyKruqvr6jr++QZj/Oav"
    "1T/ffDzCrzdPD/dfDm821Zv3t0+HJxi5+eebp8Pz58e37x8+HMwVd58eH07P1fPD6f3v1e1T9fzr8bb6pXre3Z5uj78dVnS9"
    "Ox2efr99hLebCq9/Pb478zHeVHTtnnV/b269ur/99O7DbXX6a7W6e7o7Pj3fHt8fVqcNfP3vh+PTw2lt3t4+vX22f61O693z"
    "w/3d0/NqvV6vLOyr2031bm1ve/j6eHj/fPjwFt6c7P3/fvp82FQ3NwgA21RkD+/Nb4DDvDW/NxXb79dv/rWpeuPtYbm5wcje"
    "BcENAtLpM/cYf/slcMYAAgYYsFkVZiE1QMEIgRFSjvaWdyHMl8ITFSNilw/bmTEvtHMdEVpsAZEDEGH3y5EvohbfPWD8BvOJ"
    "tvLD8fC0WvFNxc0W9bNAqp9rH63HTsMg7G/IDnZteLGLNHbAjPz3tle67cBN6KWDYVAfnr89Hn553n28f7h9piTJC+J4aMem"
    "NE9FDowtjiDMS5/IPBdnE4I8VA40O89bYf7cSvuq3GVY2e/YC0nJ1H38fH+/MsIU/jNAgu5Qe4rQjs89IXjHzYRkLwNRW60i"
    "XRDWpovqL6BdVP9b5SRiBCxMo8c/u0EXYYUbYFG/wex0g7DmFBNuKQVRjQThnoIwllhJ9wnmaZxookX4gPL8I87iB2bQEh68"
    "V1LZG9sPtBTpG5ikTzBLD8dSSsqouxfW+UckjqcnEK0UUR4NikV2OZUYaf+FHD3GM/yYJBlQnEaY7GCSckzrsVLuawf/8LuW"
    "mvds55W4O7g+vxSjsFnrm8mt91dYb/hGku+dS29UsbjZDQOwjyT2mT13h4P/++H08GQolnSCapkMHglpvIkBtPuPYUAnBfHc"
    "HBtWNQJkbvhMGUg4snB8Hricy4+dWSsuqZtDS9scoWk0uEgbyQBzQ7jJ4DJW1c2pHEscIpHck2qy2sMz7Mn2DgVP5k2chz2W"
    "bwrRtXPbwNiswGCk3YxbihCjCeL08OfTpnr/cG8o2Sjuk1sv6Qk9DW9vCRtGP3agt0ZRmwazUq9iGgrhN2S1JPx2HdxLGQKA"
    "u3gtdAhbokVXZixqHGq0Wf0lN3IMezodHs00nCPCmxzVR4PqyuutgiXEq48Pp+qxujtWp32Ygy+bytx7fU7u1CWYlT5O3wkW"
    "y2QfvfhZT5r5UhO+xNgRTqWJM0jnn8EEhjV7O/9o/FWErqwhiudHVHpEZDEKZvWtwgxrnyNDlqJ7lIxtinL6bX7wwidRV8Nz"
    "2SFPMFWHDkMEo/OWiP3OZTU0GTTmocj+yRAarNdbv6C3Rdrg9lGb+wGMMl9PVNUz91Op7s/M99XZ+T5vlPQDmwzlfG3S8Bif"
    "AXA7yewmRpbTOZlUBaUN3asluy+SABCA+18X2/4vYsW36D3TTcWQTgFc2dNZQ60b/vRMMXPaWaFyRkZPQlibSEeEoKn5ZVr3"
    "TMpM5+b8ekGNxP4AjdLeSnCT64CCADeQlmJmgzt/aiHg1PnazSufEXJZCu9WepIEvLcWeOOStkjAbOgZUTDPk+nIjvKJN4Xd"
    "Flt3gmkNZG/vWBtnK8zD3ZiZCuKnhPjzCfNVNyvwTu331zjrvDRBN8Jwb3ijghE2YMHjeUHt2IhaRkGWwcVyCGVJziy9UTPE"
    "MLTyY4x1ta1ELYaCWQFIF0Bui42BI+z8YlK4VPXDF+dFwwsggzw/CBKFDaa+/LhqXf1PhXX1l2qrWweYy1DilurA+ig1HEIk"
    "BwdHdGHRfIoYAaqfH+8B/Z2dtB4YWoZhmDtM8LpQPvBrwWbYLuzSUrhwosrA0um1QA2KMwBMiwGOyrm8FpgyzagnXvYjE++N"
    "lXXCehzsIcKrI+Ebe8RrIwx+BDJ20Vv2qEDsHTGXA44uKufjYJW1qXUqIVLX8CthlDtKUOZlcl6t9a/Hfxjjz5zku2N38yL5"
    "fjLFeFP946Jj3oWK8A5QrTeVuFPjIUI5SWWKUEcQyBYeh0wYSKYyYnMlfR0zssVq52InzDsX2mHemQAMq6mb8AozZgI5XKiG"
    "i6GAa1xohb/YvjHf35caUTw5oeCXjiRUIx7vkpuZaiJExZa4jWewFCcbGM284C1Ioqk42n/yvUtWZ36z4LRBk/gj2krq9wu4"
    "O+0fLNC+i9hAKNj0mf9JLoGHDY0xsAzDw4UMGCXAxPEFTnV3NIcfs+OwHbUSWxr0QLLIAmSmDb+GNKXNU4cur3uPwwwSZIoL"
    "erNSpvSMIEXgstEQkXGgqFw2lMCRn3eMC+aKag4eMSl+NuwipbjHwJglm5gx02YssfbSIBxRTSWQLjLk8MghbCBTuQR9IeyW"
    "qKRxgZKtZ8PKhMymiNcYmYplDKn1QawEhbDcdD0T9gvFImrW9eMDly7n4C7Ck9tXGmPZyazw+/jVkN4y8TZrnbeLWZG7cUpe"
    "OHNFMRi0bMWYDTchjlcFg282ZKzhBtTCA7XIctJTlrvOtw5o6KRnHE73ziso8NCvzuhBm+p7phP1xNPsacfZjKmoks3orciY"
    "HBBTB3x2gGOOqhnvHxlousAnB/gMAP87DIeMgSymP/o85JV2dlTU2lE8M25zHLknDklKKTkIo/Itf7NlzUA8nkeikd2s+6gG"
    "jeNExGlNNXCKMBVeE/F60bxiNOr8gASLGV2FmHiK3Mp5UUjRbsW8zmzoPllYNjv0NTA7x6LMm5he5RKmwl+iwf1e+6WB9eaD"
    "LqEr6LCeGWfZXYFx45pum2VqxSytwKH9LxZV3jAgVN1SDWDwkKkVwIhPF+GjoFnHdC//asDIBAFG4prOb3oF/v+1H/+vJU7h"
    "EW5scsaL3XRi8ylTGfsgyTAjmHKqsNBKIGVxLh/0+0IiCUQhhUaKaYqY3SPlg63M3BhN2nWD8jG31G1Ey8eKXdvewmE1J/mc"
    "xI81YpoooRihTJBwakHB3KZIcaEQQlK789Yd5UpSuI5IgoDPyWK0g57gnOnRW4eCFC+ZhaaM+nS4Pa5uv949/YJygdW46un5"
    "Q+ui3lO4umlR6ggaSoyWT+K0yvgSP3O8RrqP10wWzP3dp7tnExK8Q+eWon6kuqnujs/wyNPD5+MHGP14f/sM87xqZApUPxvj"
    "q/pbijI67Y6fPx3uV+sGiHBxplNsKgvRuQUJgMPMYk5AjhVqgFmQdQi6twdOYTrw+ekYrBZeQCluUyOZkeASCYSZoBQn/ks5"
    "RxxLzjX84ikMvQz53ErxWQy8gT8oEgvg77Svhie+EEmSGUIxs8AEJFh8SXO90QILjneEMMmY1EJyU05AWincHMzlpVv69tc6"
    "vpWcdkhfXaEjyyh0PFPoxsQlcHTerBJ1hQ6UvAntqq8l0siKbuxtlSnfeH6DeTCErvKuW/97+cUbUsGUmfSNN71ww2k24nc0"
    "EuODkrlYKw1if8tgByYTsmVMRkMwNyujIRftyviYZGpmZULsazQHo/mZyphkhmgyTqNJGk3LWhmRupkanxsN1mihZqYrU3VP"
    "ZJymZM7GaYofxWlKJi5qG7sRlGT2xmmKBnCcpmQKRzT9a/x6Zh6XqMDSu92Sx2ouFtkofjTOcvHCk8/M5Wk6xM4cZy6XCatr"
    "eKx5JtT4Qk7qKNJGeQqX8FF8/6+P4sfxUXzPapfZjAh7YjirOzyb1Y73o+ybVh04Ov/ZuXtwbV8n2GrcTE0cWBisH3egPmdY"
    "h5fquPjsOT+Zc74jvMBhesIkqARoDFIyHITOGA3gaaxQfmDU71h9Wnuo6JzJrxUOP5oqzanlt8T/UCUlqKvK6q3uh1EiwGJG"
    "doz5Hyk5V5gKu/buRxHQ2TlS9jrhfpSA+ytguZY83I/GiFDCmX2ucj+ac4wFkSSeO3ce2mfxTunIPteia5qwUXajE1HLaQrj"
    "BEKNRS+Mc+Rw/O3594m9I182lbvvBfoVQ+PevnTX3vNBCKiJ0vwYGTtJEaYI0BhWDEtDZtvOUSAgLJgWMISoUJTJYXu5PhfW"
    "TZIjT6bz7vVF3kYeDAG/jBXBMGypDFVZsM4r+5DV8XH39I/T86rBraqffwZdaff0+ZNhUpvqSwfL6jkrK7HT9ocTCmxCMeIM"
    "e0GYGWWIAh/jnDhOZscag0rWrnQGu9bNMYZJ6zomWes6DvwGIeBjQIiKS+2sbPtTHwMe1ByTdiiNZQcdglzDVKNtdXE+va4e"
    "NT7SVsP9ozronEhiJ+0kFpRK4iyYLukZos68cEQcPsESnxW2jqIQ1wJEKwvf1vZxXEiQolyTc9+O0WwU2X+YIIEp0mdlOxX2"
    "n/GOUy7dqYr5ttl1INxBM0CSn38cfIUREPiCS4UEG3cg0SjRPT/F1l/3xekxdD2/odiyEBtGlLiWKjSZsLyoIXQxW+XGQK8l"
    "SBNdqgSQ8wXAronTtltp7tpqzesKMZXNQmHXRA8P1MryQ6fzTD9Ly+Bep5nQyOrUWC4bWVgrWBgJHBSokWHHMcEuomDCIGMo"
    "gRHklA1t6pmDNoPNCT21g5xoRgTSUgmtJLXLTbUEvUdhSRhninox0KXndOo0HfpL1/Z5QS/h9BpZQ2xJvaQPA5lCZSGot8rC"
    "ZlVZFAHtA14lU1Iz7piN4BgITHCmJVcauxMmKQkxJhewH4GpOwVCRIP+gUAjx1KRUDrfBE8hSjQBgwzu6c61mOJA+3AhkD92"
    "3gZOkWFfoO9q5vyzZkEEk4pxIsDKg6v9CRgxV1EBCwIbgipP0ZRjUG+MwwJAcoySg4Zi3A+AEJLKbya4F4ZBo3DBl7y3mjGh"
    "mSZwB+CqcF8LElMYANcYcICdy9wgx9yGmgDkgAchztuBkIZrDVSwIXVoD6AEIbBNlZkPTt0tOVAZpxgjTIjzqXAM025DyRSh"
    "oNi52dCcCwxrACoh0apQ2bCu6X8Xr3SrdiZZyitdx62POO4U0VmUybVTw9jsx3GUmR4asFWAsoVSLoYBzAhz4KIRNr0zcNjQ"
    "jVGjajW/Xko6We5fyJEzRb/m9CbHIzJlgkg19wYTSmda+QfhVApMJmCemNtDtvbgmATOUBN53uPZosi7Hzd9A8N+ZxiBBBPa"
    "tYax68gJCA1Q+CglWFHlFpcqIWFNwaYHaiDI1d0mTIFtDQKDECOa3NcxAp0NlDYQQvAdRv0JrgY6AgkDFAT2PzLqJQwqkGQc"
    "mWBkJUxMih0EgWmlpFbUOkhd1CAIdNvGRyrzsfs6bEAEMl5geIxth+OfRSSIWmS0Uc0MN3PBSR4FGJOGFUYC7/qgeQfPBZrP"
    "c5K1CVkXCl24tifFEWBzBm38X2uqu9aka/Haq/yfkOQysepfT3ec85hSAyUg4KmgfDEhlSU5YsSRYqDswsfam1uCEwaEA9MP"
    "lgEjhdwXCCUUPvdhoIj8e2O85Qilw9Q5z5/HnDfzWY+akxrIJk4k7upf4Yvu/3q8f3j7+P55U/1+Z37bk8pNpSeN5K49occJ"
    "X9213QJdAPfWWoMkwArIFfhW+QH9mWwPINd2tgfqmi0AT/PXkvbRa6Id1qbForGHOcjVsv3BY3QJP0NJRpxIvhQlNeAr49Tt"
    "pi9diLr9dG1lcxj+ncDXTOaEUgwIpRMXIaWunZh95QtU4XZNkMAU17ZddFkAr9GDua2gaaqJ2UIyC6BiS75bdKj9X1r03srw"
    "uYEfoDjUymrjCQoqDYcW2wkWruuPOVi2PcaBiqjIyvhMfSBf3yu+zKylN2EbUMxff+6GhjZirAYSsSCJgSZ8suCVPY9JstUU"
    "lF8Au2arNF+wX0aGYS6I/eYt72SOHowjD5uUMRM+rE3qVrsT/caKP2JK58tih6eOzdz5IgTQeH5pOUvHtBapoXjj+r/zWis8"
    "MtHeDdWTxlZIGo1kqExd+z2m/qjdA8ssmG0tYpyz5bVpsrLnrocI8k025+cxrlCSL+ZdXCCJO91jqzMmvAg+xlGpYnHaFDYy"
    "cSIRDnKGDdAbV+8eHu431fFxZ968NYGF5s0AM9y8li2RF97zgfp/t/dPhbAGdvgDTKv+EYBMFhU2sajz04BLf5lWpmFSE2ps"
    "Mf2MerWMOZ3M6FTE91UyKhiRti/RCFFhGGsQErb641KqNkmtoXngriU42ZtIV71uGSHRAKCsvIjroeT+L4LGmT7KmOlputtH"
    "J+T61+MfLzQBLkfxdlP9cbnIe/P/fljj+Exm+z1qbB6+91iRJbAKKlIZJtyuurKl7x0SeAkkTHdDw5BkIRrEtuhbEAGe9oyY"
    "Okww627kG64l1qmXZBcNMEotsOCNcLtqEeOl0L3DkG0bh+o+HlUUjjGtj8c3zBTWvjQ+nnrn3atdMbL9BFmo/0fz+W4rUzmN"
    "9Dtz6v/r8V2jHnis3/5StFbjzCjxqHdnkDTlq4q46hbH2uQkVJVvg01iqwNfkwtPCby5WxHw9ky7CeuWTAsbLwSOxPP6FoQX"
    "cy4Gg4kGAZk3g+iVciA8DquCb8L7h/v7FZ6O1m1kUVRf2TxRpq104iweY7oj4YvJJY61sPIsftozTF+4apCFGSozTokJMMvK"
    "R6XiWQqF9ziEqbkAON+BbSdcb5Hy9D0eEsrPkEgMVJmbTPiOj62EHgul8A7M2CJkv1WJb/rdT/C04vv87n/Xijh2ItyFHU/d"
    "A+gF9hfnxD6W090wC7c0Y6ctAAZ9/ZIUuMYM1VsKsVR4jur0Xgrt3+Md8e8LLdQbkhf34V1kw2IVni2Zm2CGUkptV/A6U+jc"
    "E6RWm00vgJ9LgoiVMVyzuX+rjD6+Q6Xml1dIU9s0mldEmbEpCTObDv4Rym1yvxqZP7XMSqDylZBzzjY87DW2USpDBoiVc5D6"
    "pqwoplowFbc5Zq/Gks9Y1rTWOs4KF73rDIoeBggqt23bINBmhPmlp8sxs9BQrtvzkZXJchxvEGx2hbyF+R+sY9oelxyN0DGH"
    "5Ld16JhDvr6Mjkmwz5HyGdsqFS9WGIWyxSbIP9RsBsuT4GENK9uKZhfB0EnlSg/s5WCV8vANBEaE/EzEWZZBAIs8r8Xg1XOV"
    "a5B46nbFLR2ybQzc3LjEii/Geqo+PpyqL9XdsTo9/Lm3f8Eb83erXe7+bL9cV0UuLzk6qo0vujrI6TFlYNaqhl59evN2goUA"
    "i6wrherufXpdsnBZ/rGJsh5N93effmuUiXKVouwnLuqurlhwK/TyX3t/sQ2c6Lp4+lQ8eNylw6T07LihpDA5h6ZQiKvD3/+g"
    "LMxSJIQbjHxvTBQF8H5W/DAf1PXWYZAdDqB2UDyfgLX7HLRmLyefkHb2iLA34isL6NolOyvs6glJgTm8N5TrxxRWCmFTAFbA"
    "iK0mxCUhJrF6U2l/EVeIKkptb5naiD0I5/6LnJqsNeHi79tjBHeMsY4x2RxzbYzaV1LqxoBWg0JCRXvIY6EEERIpWzuGkeaY"
    "jWTjrS8z1f4yxx1jrGNMNm43oNNC2EepgrkhObMAasIy5n12D3FdUYln5i/+nUp+s4UMDZoZGnx6R0oPxavdnIkMYT+3HYXt"
    "WHDDdp9mb9PncyuYKD25SPuX3b7liTsL91s1vi+1DE0eUk/3Utfpw6CvL2EZ3mR1LFKVjNhrJ2vN3Wqsk+pfZHUvaKvZTWxI"
    "E7UN77AQE3vCbOxhk96w1U/M+PtmOQiruFwhxMfM+qZ6fznfEbsUZhc2GcEp2WwxiLSFvg/lbeGfgjWXQT9GbzXfFM6AbOOu"
    "2ljrxfANYWmFYb7+EJ+fcal6T+f7hqfTdi+b4hS8mMhj2JYHBrvO7ijC5XUKPTkvoDbbpiVUbWSnuNKEXN7y9cTXQmLIgts6"
    "qUHuCF8IQxQX3J3YeVBK0NQdIWl6GbS0zQIcw5oDE25Ro1f/l8ELBcZLAxPGDpZQmUL9EGcKN/lUPN6entNsWG+TGbLup32P"
    "ubk5A2UwOlpQWUsqnsLl19Co4Ker92XKdAzXyADjwTwQ3ZPoYv+2BMXO9cmp6pr78fnm9CICsafvBYDzjnNBlnR+sX7/6PaM"
    "N9oXB5m4O7gT1y4LhzlC4Bcs1OnJtgaZiYRpQOJINb8qzFC6Zl9+1pM6xW/PkaT/mPZoyzj19HRA6GuANSGKlj7OLP1whQvd"
    "4SOmSp4hGzLnbMgsgNIiLFO41z6WW9HXyDAPyX0wBfbz6VPwvm+qY98+Y/kL6RijHWOsY6y83pUxDeJk4EUmQ0Y0VDEaUan1"
    "mKBFMGE1IVycAO9x4IvgoDPLSI/O4bu/fXe4f2rqpCRY29dKg3WPvZyvWN+I9S4puKv3ZOuK3qvcMROZf8RMA1tuGmpIdrfd"
    "fGl0zCSg6yWsDqGCUTgQ7/76o2jXjjyw64mlPVkOx3YZBXeU1+/o4ZsVAb5OxUAapDM+q4WYWpqniyXdaFFcAQ/lZ7hNoR4F"
    "ASmCAFnFxL+OhAAVQSBGPlUUPVU7p5hVQ0Y9nkXFcXRThW9nBNZLK/PC/l799PGn6u5jcyvfPb21pzJ3x9/ePj4YJNcV7ORD"
    "9dPdT302/rcLGx5u47PUrXs+ln7AKYG5ORqu7X+e/K37KGByXv7tIg+PhesG1EGrQ0+XgDtW4SgGGxeS5Tix862HuCHrkJEf"
    "iCsXJGTqtFS0c+X/s+6KNlrLdcOavVkMPidXr94/ik1QeqGwM06H8jzZR+XF9IJ7wbjSF2gbhMe2QaKLNG9p9G5xu1aSq9R1"
    "yQqpIF+tzpUAc/8XqQfljidYaVUUYqqi8HqMn1ik8C3KzrgHFZJuFqe11Z4WKbaji1PQk8JDlwDcS2d/qIXEtLvHVW7Vdllt"
    "FMLxKgW4jOPocozB0DXKejIgDzpfAnQTdgrQay3sq7Svyr7q4vYn6RgaRA4PS4OXwG+Lx/hV1aJk5QAIWSZk6u0TpbPOshhf"
    "LGJ3ukH7rG/DCcyNTsTOumNqfdjDkXfB0hAb1ONsZ3TRlToebJTqVTpHRXEzpHgMesnzNx5o+xhrE5Y2bnLRcXSOGc4eloLU"
    "kual5Nz20rXqOX7v45L2zvHolcD7a9fzMSdIS+Oc3AFJb5lubETzSM8nQ9jBH9cSZn/0qLqIxhTLzcPodRMlshRK9SL+bsPr"
    "6R0kLG54nPUKJC6/bfFJQL7YfbmHhL+w1QWrbfUrVuHtg6624tumIPV+W0ry8nXs3PI929HC6nqnpn1QoeFMIHONEDpxDjSt"
    "d/FzNgUen8ZpPvq6NjpR7wZJW97ZUXBX2pMqKwxHWlG7S+CH87pUASr7IouRZDtyoQ7olfFhtm0i2LJIE8ExlYWI6DxZLBY4"
    "pYsgRlOJVfvCvPWWxCZVk/ZFIKhV5u2xaSOnKOaSg5hTdfdUfa9Pzfd1d0PAxwvHLlGa1hpW1UsWu/RK0+LK9MgzLRa8UeeL"
    "GPtmC7Eh1oDjwA7fri/YzSzHfpzk5OoaExYKOWPXaGe0h7SJ66tBNIFY5q2rnzPRFqKZaHw9GHtChr98le597BXJc2M7HU/y"
    "ibKCHo6Hp6yci/+z+rlAy3aTc1ufnNuuybmcfpEJipga5XK81uWlnLfnE4pTLaptWefrqyBfq4doiw2ti0uEd1X7ei2I4p0t"
    "2Vlc2rJf4kxUDvYh+6403/oac3CzdX3tt6FSp3B/0lp4GydyWiV+m5RmGZoJm36+JkjZhwuf9SjuPtw+3759fD6t1tUvv1Rf"
    "s7831amjzalpGGxu3ifFngwNHO8q0LTNHSnYk0VE0OG+BIapcGfMnRoUQfC1q1Kea3FTXz68yPJFcIrQcUWiVYMaXXwEWQCf"
    "+OSkw+OJdqFXWiQg+udVXD2A4589W5LsUxau93MFnTNTREIiQ6nJ6bqbaR9d/+dVvHl9cLaAuFaYvmWK9h0sC8NzhT/gXm4Z"
    "AxCFOLTyj/68iiOuDyZ2YZxh5tqUxna/vneTmLTfGQmn+3TC2iLHPilWNZdhSs1JObH+4Nd3qnBKqsu+cx4u+zXs5If9oisp"
    "j1VxoXAWnyySD22aA/Yh01ObhdKuaWlyCE9ozLrKvBh6GRIrYwWLKQI0hgDfhCBkXelJZR/uOBKZCsmvk2zCQoZK4430lJUN"
    "v17eOaUSIPmY49v54AYD1/0wSoRkDClrrDUHS3GrJaM6hmAwRHOiyBMvFzthfziSkiDNZBbByif1DysjyjfV0/PB+MGM9+sa"
    "J1L+CQNUzUy/HKxUfm+1K61hiF8Hhr2V6dSfr+EBHNmvr9b0z1DevjZTJlxqwZkKqDvISgOgQomYBgnwBRELMLkt/a//B6s+"
    "Oj0="
)
print("Delta Drills checker ready — 67 problems. Run dd_check(<problem number>) under any of them.")


<!-- dd:dd-kp-numpy-broadcasting-rules -->

## Broadcasting rules

`numpy.broadcasting-rules`


<!-- dd:dd-seg-numpy-broadcasting-rules-0 -->

### the right-alignment rule


Elementwise operations "require matching shapes" — except PyTorch will
**stretch** certain mismatched shapes to fit, following one mechanical rule
set called **broadcasting**. It is the engine under half of idiomatic
tensor code,
and it is fully predictable:

> **Align the two shapes from the RIGHT. For each axis pair, the sizes are
> compatible if they are equal, or if one of them is 1** (that axis gets
> conceptually copied to match). **A missing leading axis counts as 1.**

Work `(m, 1) + (1, n)` by hand: align → axis 0 is m vs 1 (stretch the 1 → m),
axis 1 is 1 vs n (stretch → n) — result shape `(m, n)`, where entry [i, j] =
`a[i, 0] + b[0, j]`. Every pairwise-combination table, distance matrix, and
outer product starts exactly like this.

The stretching is *virtual* — no copies are made; PyTorch just reuses the
single row/column while iterating. Two everyday cases you have already been
using: scalar-with-array (`z * 2`) and matrix-with-row (`z - row` where row
has shape (n,) — aligned right, it matches z's last axis).


In [ ]:
import torch as t

a = t.arange(3).reshape(3, 1)      # column: [[0], [1], [2]]           (3, 1)
b = t.arange(4).reshape(1, 4)      # row:    [[0, 1, 2, 3]]            (1, 4)

# Align right:  (3, 1)
#               (1, 4)
# axis 1: 1 vs 4 -> stretch a across columns; axis 0: 3 vs 1 -> stretch b
# down rows. Result (3, 4): the "addition table" of the two vectors.
table = a + b
assert table.shape == (3, 4)
assert table.tolist() == [[0, 1, 2, 3],
                          [1, 2, 3, 4],
                          [2, 3, 4, 5]]

# The result SHAPE is decidable from the two input shapes alone, with nothing
# allocated — which is what you want when the real tensors are large and you
# only need to know whether they will fit together.
assert t.broadcast_shapes(a.shape, b.shape) == (3, 4)

# Incompatible shapes fail here too, and fail early.
print("(3,1) + (1,4) ->", tuple(table.shape))
print(table)

try:
    t.broadcast_shapes((3, 4), (5, 4))
except RuntimeError as err:
    print("RuntimeError:", err)
else:
    raise AssertionError("(3, 4) and (5, 4) should not broadcast")




Why: writing the two shapes one above the other, right-aligned, and
resolving each column IS the method — do it on paper until it's automatic.
The result shape falls out before any code runs, which is exactly what
`t.broadcast_shapes` is for: it answers the shape question without touching a
single element, so reach for it instead of building a result you intend to
throw away.


<!-- dd:dd-q111 -->

### Problem 111 · faded — your turn

The broadcast sum of a column (m, 1) and a row (1, n).

**Expected output** — run the cell below once `solve` is right and it should print this.

```text
tensor([[0, 1, 2],
        [1, 2, 3],
        [2, 3, 4]])
```


In [ ]:
import torch as t

def solve(a, b):
    """(m, n) table where entry [i, j] = a[i, 0] + b[0, j]."""
    return _____ + _____


# Example run — the grader calls solve() with several pairs.
print(solve(t.arange(3).reshape(3, 1), t.arange(3).reshape(1, 3)))


In [ ]:
# Did it work? Run this. (NameError → run the checker cell at
# the top of the notebook first: Runtime ▸ Run before.)
dd_check(111)


In [ ]:
#@title 💡 Solution — Problem 111
# Running this rebinds `solve` to the reference answer. Re-run your
# own cell before dd_check() again, or you are checking this one.
import torch as t
def solve(a, b):
    return a + b


print(solve(t.arange(3).reshape(3, 1), t.arange(3).reshape(1, 3)))


<!-- dd:dd-seg-numpy-broadcasting-rules-1 -->

### placing the 1s yourself with None


The craft skill is **placing the 1s yourself**. Indexing with `None` (alias
`None`, which is what NumPy spells `np.newaxis`) inserts a length-1 axis:
`v[:, None]` turns shape (n,) into a
**column** (n, 1); `v[None, :]` makes an explicit **row** (1, n). When an
operation needs a vector to run *down* rather than *across* (or to hit a
specific axis of a 3-D array), you reshape it with `None` until the alignment
says what you mean.

When shapes are incompatible (say (3,) with (4,)), PyTorch raises rather than
guessing — a broadcast error means your alignment is wrong, and the fix is
almost always a well-placed `None`. Never reshape at random until the error
goes away: work the right-alignment on paper, decide where the 1 belongs,
and place it deliberately.


In [ ]:
import torch as t

# The addition table again, from FLAT vectors — we place the 1-axes:
va, vb = t.arange(3), t.arange(4)
table = va[:, None] + vb[None, :]
assert table.shape == (3, 4)

# 3-D case: image (h, w, c) scaled per-PIXEL by map (h, w).
# Align right: (2, 2, 3) vs (2, 2) -> trailing axes are 3 vs 2: INCOMPATIBLE.
# The map needs its stretch-axis at the END: scale[:, :, None] is (2, 2, 1).
img = t.ones((2, 2, 3))
scale = t.tensor([[1.0, 2.0],
                  [3.0, 4.0]])
scaled = img * scale[:, :, None]
assert scaled.shape == (2, 2, 3)
assert scaled[1, 0].tolist() == [3.0, 3.0, 3.0]   # whole pixel scaled by 3
print("flat vectors, 1-axes placed by hand ->", tuple(table.shape))
print(table)
print("scale", tuple(scale.shape), "-> scale[:, :, None]",
      tuple(scale[:, :, None].shape), "-> img * it", tuple(scaled.shape))
print("pixel [1, 0] scaled by 3:", scaled[1, 0])




Why: the `va[:, None] + vb[None, :]` form is the general recipe for "all
pairs f(a_i, b_j)". In the 3-D case, the naive `img * scale` FAILS the
alignment check — working the rule shows the 1 must go at the end.


<!-- dd:dd-q151 -->

### Problem 151 · faded — your turn

Scale each pixel of an (h, w, c) image by a per-pixel (h, w) map.

**Expected output** — run the cell below once `solve` is right and it should print this.

```text
tensor([[[1., 1., 1.],
         [2., 2., 2.]],

        [[3., 3., 3.],
         [4., 4., 4.]]])
```


In [ ]:
import torch as t

def solve(a, b):
    """(h, w, c) image a scaled per-pixel by (h, w) map b."""
    return a * b[_____]


# Example run — the grader calls solve() with several inputs.
print(solve(t.ones((2, 2, 3)), t.tensor([[1.0, 2.0], [3.0, 4.0]])))


In [ ]:
# Did it work? Run this. (NameError → run the checker cell at
# the top of the notebook first: Runtime ▸ Run before.)
dd_check(151)


In [ ]:
#@title 💡 Solution — Problem 151
# Running this rebinds `solve` to the reference answer. Re-run your
# own cell before dd_check() again, or you are checking this one.
import torch as t

def solve(a, b):
    return a * b[:, :, None]


print(solve(t.ones((2, 2, 3)), t.tensor([[1.0, 2.0], [3.0, 4.0]])))


<!-- dd:dd-q499 -->

### Problem 499 · guided

Write a function solve(x, bias) where x has shape (rows, cols) and bias has shape (cols,), returning x with bias added to EVERY row. No loop and no tiling: shapes line up from the RIGHT, so a length-cols vector already matches x's last axis.

**Expected output** — run the cell below once `solve` is right and it should print this.

```text
tensor([[11., 22.],
        [13., 24.]])
```


<details>
<summary>Hints</summary>

1. Do NOT tile the bias. Write the addition and let the shapes meet.
2. Shapes align from the RIGHT: a (cols,) vector already lines up with the
   last axis of a (rows, cols) matrix, and the missing axis is treated as 1.
3. `x + bias`.

</details>


In [ ]:
import torch as t

def solve(x, bias):
    """Add a per-column bias vector to every row."""
    return None


# Example run — the grader calls solve() with several different inputs,
# including edge cases. Your function must work for all of them.
example = (t.tensor([[1.0, 2.0], [3.0, 4.0]]), t.tensor([10.0, 20.0]))
print(solve(*example))


In [ ]:
# Did it work? Run this. (NameError → run the checker cell at
# the top of the notebook first: Runtime ▸ Run before.)
dd_check(499)


In [ ]:
#@title 💡 Solution — Problem 499
# Running this rebinds `solve` to the reference answer. Re-run your
# own cell before dd_check() again, or you are checking this one.
import torch as t

def solve(x, bias):
    """Add a per-column bias vector to every row."""
    return x + bias


example = (t.tensor([[1.0, 2.0], [3.0, 4.0]]), t.tensor([10.0, 20.0]))
print(solve(*example))


<!-- dd:dd-q500 -->

### Problem 500 · guided

Write a function solve(a, b) that returns, as a plain tuple of ints, the shape you get from combining a and b elementwise. Answer it without building the result: the shape follows from the two input shapes alone, and materialising a combination you are going to throw away can cost real memory on real tensors.

**Expected output** — run the cell below once `solve` is right and it should print this.

```text
(3, 4)
```


<details>
<summary>Hints</summary>

1. You are asked for the resulting shape, not for a rule recited back.
2. There is a function that takes the two SHAPES and returns the combined one,
   without building anything — then convert its torch.Size to a plain tuple.
3. `tuple(t.broadcast_shapes(a.shape, b.shape))`.

</details>


In [ ]:
import torch as t

def solve(a, b):
    """Return the shape broadcasting a and b together produces."""
    return None


# Example run — the grader calls solve() with several different inputs,
# including edge cases. Your function must work for all of them.
example = (t.zeros(3, 1), t.zeros(1, 4))
print(solve(*example))


In [ ]:
# Did it work? Run this. (NameError → run the checker cell at
# the top of the notebook first: Runtime ▸ Run before.)
dd_check(500)


In [ ]:
#@title 💡 Solution — Problem 500
# Running this rebinds `solve` to the reference answer. Re-run your
# own cell before dd_check() again, or you are checking this one.
import torch as t

def solve(a, b):
    """Return the shape broadcasting a and b together produces."""
    return tuple(t.broadcast_shapes(a.shape, b.shape))



example = (t.zeros(3, 1), t.zeros(1, 4))
print(solve(*example))


<!-- dd:dd-q60 -->

### Problem 60 · independent

Write a function solve(rows, cols) that takes two positive integers and returns a rows x cols matrix in which every row contains the values 0 through cols-1 in ascending order — every row is identical, and each entry equals its own column index.

**Expected output** — run the cell below once `solve` is right and it should print this.

```text
tensor([[0, 1, 2, 3, 4],
        [0, 1, 2, 3, 4],
        [0, 1, 2, 3, 4],
        [0, 1, 2, 3, 4],
        [0, 1, 2, 3, 4]])
```


In [ ]:
import torch as t

def solve(rows, cols):
    """Return a rows x cols matrix where each entry equals its column index."""
    return None


# Example run — the grader calls solve() with several sizes.
print(solve(5, 5))


In [ ]:
# Did it work? Run this. (NameError → run the checker cell at
# the top of the notebook first: Runtime ▸ Run before.)
dd_check(60)


In [ ]:
#@title 💡 Solution — Problem 60
# Running this rebinds `solve` to the reference answer. Re-run your
# own cell before dd_check() again, or you are checking this one.
import torch as t

def solve(rows, cols):
    return t.tile(t.arange(cols), (rows, 1))


print(solve(5, 5))


<!-- dd:dd-q81 -->

### Problem 81 · independent

Write a function solve(v, reps) that takes a 1-D PyTorch tensor v and a positive integer reps. Build the 2-D matrix whose reps rows are each a copy of v, then return a tuple of (that matrix, the per-column means of the matrix). Since every row is identical, the column means should reproduce v's values as floats.

**Expected output** — run the cell below once `solve` is right and it should print this.

```text
(tensor([[0, 1, 2, 3, 4],
        [0, 1, 2, 3, 4],
        [0, 1, 2, 3, 4],
        [0, 1, 2, 3, 4]]), tensor([0., 1., 2., 3., 4.]))
```


In [ ]:
import torch as t

def solve(v, reps):
    """Return (matrix of reps stacked copies of v, its per-column means)."""
    return None


# Example run — the grader calls solve() with several inputs.
example = t.arange(5)
print(solve(example, 4))


In [ ]:
# Did it work? Run this. (NameError → run the checker cell at
# the top of the notebook first: Runtime ▸ Run before.)
dd_check(81)


In [ ]:
#@title 💡 Solution — Problem 81
# Running this rebinds `solve` to the reference answer. Re-run your
# own cell before dd_check() again, or you are checking this one.
import torch as t
def solve(v, reps):
    m = t.tile(v, (reps, 1))
    return m, m.to(t.float32).mean(dim=0)


example = t.arange(5)
print(solve(example, 4))


<!-- dd:dd-q501 -->

### Problem 501 · independent

Write a function solve(x, scale) where x has shape (rows, cols) and scale has shape (rows,), returning x with row i multiplied by scale[i]. Right-alignment would pair scale against the COLUMNS, which is not what you want — insert a length-1 axis with None so it lines up against the rows instead.

**Expected output** — run the cell below once `solve` is right and it should print this.

```text
tensor([[ 10.,  20.],
        [300., 400.]])
```


In [ ]:
import torch as t

def solve(x, scale):
    """Scale every ROW by its own factor."""
    return None


# Example run — the grader calls solve() with several different inputs,
# including edge cases. Your function must work for all of them.
example = (t.tensor([[1.0, 2.0], [3.0, 4.0]]), t.tensor([10.0, 100.0]))
print(solve(*example))


In [ ]:
# Did it work? Run this. (NameError → run the checker cell at
# the top of the notebook first: Runtime ▸ Run before.)
dd_check(501)


In [ ]:
#@title 💡 Solution — Problem 501
# Running this rebinds `solve` to the reference answer. Re-run your
# own cell before dd_check() again, or you are checking this one.
import torch as t

def solve(x, scale):
    """Scale every ROW by its own factor."""
    return x * scale[:, None]


example = (t.tensor([[1.0, 2.0], [3.0, 4.0]]), t.tensor([10.0, 100.0]))
print(solve(*example))


<!-- dd:dd-q502 -->

### Problem 502 · independent

Write a function solve(rows, cols) returning a (rows, cols) integer tensor whose [i][j] entry is i*j. Build two 1-D ranges and give each a length-1 axis on the side the other one varies along — the outer product falls out of broadcasting, with no loop and no tiling.

**Expected output** — run the cell below once `solve` is right and it should print this.

```text
tensor([[0, 0, 0, 0],
        [0, 1, 2, 3],
        [0, 2, 4, 6]])
```


In [ ]:
import torch as t

def solve(rows, cols):
    """Build a multiplication table from two 1-D ranges."""
    return None


# Example run — the grader calls solve() with several different inputs,
# including edge cases. Your function must work for all of them.
example = (3, 4)
print(solve(*example))


In [ ]:
# Did it work? Run this. (NameError → run the checker cell at
# the top of the notebook first: Runtime ▸ Run before.)
dd_check(502)


In [ ]:
#@title 💡 Solution — Problem 502
# Running this rebinds `solve` to the reference answer. Re-run your
# own cell before dd_check() again, or you are checking this one.
import torch as t

def solve(rows, cols):
    """Build a multiplication table from two 1-D ranges."""
    r = t.arange(rows)
    c = t.arange(cols)
    return r[:, None] * c[None, :]


example = (3, 4)
print(solve(*example))


#### Common mistakes

- **"Broadcasting matches shapes from the left."** — From the RIGHT. `(3,)`
  against `(3, 4)` aligns 3-with-4 and fails; `(4,)` against `(3, 4)` aligns
  4-with-4 and works. Most surprise errors are left-alignment intuition.
- **"Stretching copies the data."** — The stretch is virtual; memory is
  reused, not duplicated. Broadcasting a (10000, 1) against (1, 10000) does
  NOT allocate 10⁸ intermediate elements for the inputs.
- **"When shapes don't broadcast, reshape until the error goes away."** —
  Random reshaping produces silently WRONG results more often than errors.
  Work the right-alignment on paper, decide where the 1 belongs, and place it
  with `None` deliberately.


<!-- dd:dd-kp-numpy-axis-reductions -->

## Reductions along an axis — and keepdims

`numpy.axis-reductions`


<!-- dd:dd-seg-numpy-axis-reductions-0 -->

### axis= — the axis you name disappears


Whole-array reductions collapse everything to one number. Add **`axis=`** and
the reduction collapses **only that axis**, leaving the rest of the shape
intact:

> **The axis you name is the axis that DISAPPEARS.**

For a (r, c) matrix:

- `x.sum(axis=0)` — axis 0 (rows) disappears → shape (c,): **column sums**
  (you summed *down* each column).
- `x.sum(axis=1)` — axis 1 disappears → shape (r,): **row sums**.

The naming feels backwards until you anchor it: `axis=0` does NOT mean
"per-row results", it means "reduce ALONG axis 0" — the r rows are collapsed
on top of each other. Predict the output shape first (cross the named axis
out of the shape tuple) and the direction sorts itself out.

Everything from the aggregation KP takes `dim=`: `mean`, `amin`, `amax`,
`std`, `any`, `all`, `argmax`, plus `t.quantile` and friends. Two PyTorch
wrinkles carry through this whole KP: `mean` refuses an integer tensor (cast
with `.to(t.float32)` first), and `std` divides by n−1 unless you pass
`correction=0`.


In [ ]:
import torch as t

x = t.tensor([[1.0, 2.0, 3.0],
              [10.0, 20.0, 30.0]])    # shape (2, 3) — float, so mean works

# dim=0 -> the 2 rows collapse onto each other -> one sum PER COLUMN.
col_sums = x.sum(dim=0)
assert tuple(col_sums.shape) == (3,)  # (2, 3) with dim 0 crossed out
assert col_sums.tolist() == [11.0, 22.0, 33.0]

# dim=1 -> the 3 columns collapse -> one value PER ROW.
row_means = x.mean(dim=1)
assert tuple(row_means.shape) == (2,)
assert row_means.tolist() == [2.0, 20.0]
print("x shape", tuple(x.shape))
print("sum(dim=0) ", col_sums,  "shape", tuple(col_sums.shape))
print("mean(dim=1)", row_means, "shape", tuple(row_means.shape))




Why: for each reduction, the assert on `.shape` comes BEFORE the values —
that's the recommended order in your own code too: predict the shape by
crossing out the named axis, then check the numbers.


<!-- dd:dd-q220 -->

### Problem 220 · faded — your turn

One sum per column.

**Expected output** — run the cell below once `solve` is right and it should print this.

```text
tensor([11, 22, 33])
```


In [ ]:
import torch as t

def solve(x):
    """Column sums of a 2-D matrix: which axis disappears?"""
    return x.sum(_____=_____)


# Example run — the grader calls solve() with several different arrays,
# including edge cases. Your function must work for all of them.
example = t.tensor([[1, 2, 3], [10, 20, 30]])
print(solve(example))


In [ ]:
# Did it work? Run this. (NameError → run the checker cell at
# the top of the notebook first: Runtime ▸ Run before.)
dd_check(220)


In [ ]:
#@title 💡 Solution — Problem 220
# Running this rebinds `solve` to the reference answer. Re-run your
# own cell before dd_check() again, or you are checking this one.
import torch as t

def solve(x):
    return x.sum(dim=0)


example = t.tensor([[1, 2, 3], [10, 20, 30]])
print(solve(example))


<!-- dd:dd-seg-numpy-axis-reductions-1 -->

### tuples of axes, and keepdims


Higher-rank arrays allow a *tuple* of axes — `x.sum(axis=(-2, -1))` collapses
the last two dimensions at once (e.g. summing each image of a batch), and
negative indices count from the end just like in indexing. That makes
"per-image" reductions one call, robust to how many leading batch axes exist.

One more switch on the same call: **`keepdims=True`** keeps the reduced axis
as length 1 instead of deleting it — shape (r, c) → (r, 1) rather than (r,).
Why you'd want that: a (r, 1) result broadcasts back against the original
(r, c) *by row*. The reduce → keepdims → operate pipeline is the heart of the
next KP (centering), where you'll practice it.


In [ ]:
import torch as t

# Tuple of axes on a 4-D batch (a, b, c, d): collapse the last two ->
# one total per (a, b) slice. Negative axes save counting.
batch = t.arange(24).reshape(2, 3, 2, 2)
totals = batch.sum(dim=(-2, -1))
assert totals.shape == (2, 3)
assert totals[0, 0] == 0 + 1 + 2 + 3

# keepdim preview: the reduced dim survives as 1, so the result still
# lines up against the original for broadcasting.
# (t.mean needs a float tensor — it will not promote ints the way numpy does.)
x = t.tensor([[1.0, 2.0, 3.0], [10.0, 20.0, 30.0]])
rm = x.mean(dim=1, keepdim=True)
assert tuple(rm.shape) == (2, 1)
centered = x - rm                     # (2,3) - (2,1): broadcasts by row
assert centered[0].tolist() == [-1.0, 0.0, 1.0]
print("(2,3,2,2) summed over the last two ->", tuple(totals.shape))
print(totals)
print("keepdim=True keeps the axis as 1:", tuple(rm.shape), "->", rm.tolist())
print(centered)




Why: a bare `(2,)` row-mean would align against the WRONG axis when
broadcast (right-aligned → columns) — the source of a classic silent bug
when r = c. keepdims makes the intended alignment explicit.


<!-- dd:dd-q135 -->

### Problem 135 · faded — your turn

Per-slice totals of a 4-D batch: collapse the LAST two axes in one call.

**Expected output** — run the cell below once `solve` is right and it should print this.

```text
tensor([[ 6, 22, 38],
        [54, 70, 86]])
```


In [ ]:
import torch as t

def solve(x):
    """(a, b, c, d) -> (a, b): total of each c*d slice."""
    return x.sum(_____=_____)


# Example run — the grader calls solve() with several different arrays,
# including edge cases. Your function must work for all of them.
example = t.arange(24).reshape(2, 3, 2, 2)
print(solve(example))


In [ ]:
# Did it work? Run this. (NameError → run the checker cell at
# the top of the notebook first: Runtime ▸ Run before.)
dd_check(135)


In [ ]:
#@title 💡 Solution — Problem 135
# Running this rebinds `solve` to the reference answer. Re-run your
# own cell before dd_check() again, or you are checking this one.
import torch as t

def solve(x):
    return x.sum(dim=(-2, -1))


example = t.arange(24).reshape(2, 3, 2, 2)
print(solve(example))


<!-- dd:dd-q503 -->

### Problem 503 · guided

Write a function solve(x) that takes a 2-D integer tensor and returns a 1-D tensor holding the sum of each row, so the result has one entry per row. The axis you name is the one that DISAPPEARS — name the columns to collapse them.

**Expected output** — run the cell below once `solve` is right and it should print this.

```text
tensor([ 6, 60])
```


<details>
<summary>Hints</summary>

1. One number per row means the COLUMN axis has to go.
2. The axis you name in `dim=` is the one that disappears, so name the one
   you are collapsing, not the one you are keeping.
3. `x.sum(dim=1)`.

</details>


In [ ]:
import torch as t

def solve(x):
    """Return the sum of each ROW."""
    return None


# Example run — the grader calls solve() with several different inputs,
# including edge cases. Your function must work for all of them.
example = t.tensor([[1, 2, 3], [10, 20, 30]])
print(solve(example))


In [ ]:
# Did it work? Run this. (NameError → run the checker cell at
# the top of the notebook first: Runtime ▸ Run before.)
dd_check(503)


In [ ]:
#@title 💡 Solution — Problem 503
# Running this rebinds `solve` to the reference answer. Re-run your
# own cell before dd_check() again, or you are checking this one.
import torch as t

def solve(x):
    """Return the sum of each ROW."""
    return x.sum(dim=1)


example = t.tensor([[1, 2, 3], [10, 20, 30]])
print(solve(example))


<!-- dd:dd-q504 -->

### Problem 504 · guided

Write a function solve(x) that returns the per-row sums of a 2-D tensor but with the reduced axis kept as a length-1 axis, so a (3, 4) input gives a (3, 1) result rather than (3,). Keeping it is what lets the answer broadcast back against the original — which is the whole reason keepdim exists.

**Expected output** — run the cell below once `solve` is right and it should print this.

```text
tensor([[ 6],
        [60]])
```


<details>
<summary>Hints</summary>

1. Same reduction as before, but the result must still have two axes.
2. There is a keyword that leaves the collapsed axis behind at length 1,
   which is exactly what a later broadcast needs.
3. `x.sum(dim=1, keepdim=True)`.

</details>


In [ ]:
import torch as t

def solve(x):
    """Return the row sums with the reduced axis KEPT."""
    return None


# Example run — the grader calls solve() with several different inputs,
# including edge cases. Your function must work for all of them.
example = t.tensor([[1, 2, 3], [10, 20, 30]])
print(solve(example))


In [ ]:
# Did it work? Run this. (NameError → run the checker cell at
# the top of the notebook first: Runtime ▸ Run before.)
dd_check(504)


In [ ]:
#@title 💡 Solution — Problem 504
# Running this rebinds `solve` to the reference answer. Re-run your
# own cell before dd_check() again, or you are checking this one.
import torch as t

def solve(x):
    """Return the row sums with the reduced axis KEPT."""
    return x.sum(dim=1, keepdim=True)


example = t.tensor([[1, 2, 3], [10, 20, 30]])
print(solve(example))


<!-- dd:dd-q108 -->

### Problem 108 · independent

Write a function solve(x, qs) that takes a 2-D PyTorch float tensor x of shape (n_rows, n_cols) and a list of quantile fractions qs (each in [0, 1]). Return a 2-D tensor of shape (len(qs), n_cols) where row i holds the qs[i] quantile of each column of x, using PyTorch's default (linear) interpolation.

**Expected output** — run the cell below once `solve` is right and it should print this.

```text
tensor([[ 1.5000, 15.0000],
        [ 2.0000, 20.0000],
        [ 2.5000, 25.0000]])
```


In [ ]:
import torch as t

def solve(x, qs):
    """Return per-column quantiles of x, one row per requested fraction."""
    return None


# Example run — the grader calls solve() with several inputs.
example = t.tensor([[1.0, 10.0], [2.0, 20.0], [3.0, 30.0]])
print(solve(example, [0.25, 0.5, 0.75]))


In [ ]:
# Did it work? Run this. (NameError → run the checker cell at
# the top of the notebook first: Runtime ▸ Run before.)
dd_check(108)


In [ ]:
#@title 💡 Solution — Problem 108
# Running this rebinds `solve` to the reference answer. Re-run your
# own cell before dd_check() again, or you are checking this one.
import torch as t
def solve(x, qs):
    return t.quantile(x, t.tensor(qs, dtype=x.dtype), dim=0)


example = t.tensor([[1.0, 10.0], [2.0, 20.0], [3.0, 30.0]])
print(solve(example, [0.25, 0.5, 0.75]))


<!-- dd:dd-q130 -->

### Problem 130 · independent

Write a function solve(z) that takes a 2-D PyTorch integer tensor and returns a 1-D tensor with one entry per row: the column index of the row's FIRST nonzero element, or -1 if the row contains only zeros.

**Expected output** — run the cell below once `solve` is right and it should print this.

```text
tensor([ 2,  1, -1])
```


In [ ]:
import torch as t

def solve(z):
    """Return each row's first-nonzero column index, or -1 if none."""
    return None


# Example run — the grader calls solve() with several arrays.
example = t.tensor([[0, 0, 3], [0, 2, 0], [0, 0, 0]])
print(solve(example))


In [ ]:
# Did it work? Run this. (NameError → run the checker cell at
# the top of the notebook first: Runtime ▸ Run before.)
dd_check(130)


In [ ]:
#@title 💡 Solution — Problem 130
# Running this rebinds `solve` to the reference answer. Re-run your
# own cell before dd_check() again, or you are checking this one.
import torch as t
def solve(z):
    mask = z != 0
    return t.where(mask.any(dim=1), mask.int().argmax(dim=1), -1)


example = t.tensor([[0, 0, 3], [0, 2, 0], [0, 0, 0]])
print(solve(example))


<!-- dd:dd-q505 -->

### Problem 505 · independent

Write a function solve(x) that takes a 2-D float tensor with no zero row-sums and returns a tensor of the same shape in which every row sums to 1. Reduce along the columns, keep the axis so the result still has two, and divide — the length-1 axis broadcasts back across the row it came from.

**Expected output** — run the cell below once `solve` is right and it should print this.

```text
tensor([[0.2500, 0.7500],
        [0.5000, 0.5000]])
```


In [ ]:
import torch as t

def solve(x):
    """Make every row of a float tensor sum to 1."""
    return None


# Example run — the grader calls solve() with several different inputs,
# including edge cases. Your function must work for all of them.
example = t.tensor([[1.0, 3.0], [2.0, 2.0]])
print(solve(example))


In [ ]:
# Did it work? Run this. (NameError → run the checker cell at
# the top of the notebook first: Runtime ▸ Run before.)
dd_check(505)


In [ ]:
#@title 💡 Solution — Problem 505
# Running this rebinds `solve` to the reference answer. Re-run your
# own cell before dd_check() again, or you are checking this one.
import torch as t

def solve(x):
    """Make every row of a float tensor sum to 1."""
    return x / x.sum(dim=1, keepdim=True)


example = t.tensor([[1.0, 3.0], [2.0, 2.0]])
print(solve(example))


<!-- dd:dd-q174 -->

### Problem 174 · independent

Write a function solve(z) that takes a 2-D float tensor with an ODD number of columns and returns a 1-D tensor holding each row's MEDIAN — the middle value of that row when sorted.

**Expected output** — run the cell below once `solve` is right and it should print this.

```text
tensor([2.])
```


In [ ]:
import torch as t

def solve(z):
    """Return the median of each row of z (odd column count)."""
    return None


# Example run — the grader calls solve() with several arrays.
print(solve(t.tensor([[3.0, 1.0, 2.0]])))


In [ ]:
# Did it work? Run this. (NameError → run the checker cell at
# the top of the notebook first: Runtime ▸ Run before.)
dd_check(174)


In [ ]:
#@title 💡 Solution — Problem 174
# Running this rebinds `solve` to the reference answer. Re-run your
# own cell before dd_check() again, or you are checking this one.
import torch as t
def solve(z):
    return t.median(z, dim=1).values


print(solve(t.tensor([[3.0, 1.0, 2.0]])))


#### Common mistakes

- **"axis=0 gives row sums."** — axis=0 REMOVES axis 0: the rows collapse
  together, yielding one result per column. Cross the axis out of the shape
  tuple and read what's left.
- **"keepdims is cosmetic."** — It preserves alignment for broadcasting.
  `x - x.mean(dim=1)` on a square matrix runs WITHOUT error and quietly
  subtracts along the wrong axis; `keepdims=True` (shape (r,1)) makes the
  intended row-wise alignment explicit and correct.
- **"Reducing two axes needs two calls."** — `axis=(1, 2)` collapses both in
  one pass. Chained single-axis calls also shift the axis numbering between
  calls — a tuple avoids that trap entirely.


<!-- dd:dd-kp-numpy-centering -->

## Centering and standardizing rows/columns

`numpy.centering`


<!-- dd:dd-seg-numpy-centering-0 -->

### the template — statistic, then operate


An enormous share of data preprocessing is one sentence: **compute a
statistic, then subtract/divide it back into the data.** Reduction produces
the statistic; broadcasting spreads it back. The template to internalize:

> `result = z  OP  z.STATISTIC(...)`

Pick the statistic (mean, std, min, max…), pick the operation (subtract to
center, divide to scale). The simplest case is **global centering**:
`z - z.mean()` — a scalar statistic, broadcasts everywhere; the result's
overall mean is 0, whatever z's shape.


In [ ]:
import torch as t

z = t.tensor([[1.0, 2.0, 3.0],
              [10.0, 20.0, 30.0]])

# Global centering: one scalar mean, subtracted everywhere.
centered = z - z.mean()
assert t.allclose(centered.mean(), t.tensor(0.0))
print("one scalar mean:", z.mean().item())
print(centered)
print("new mean:", centered.mean().item())




Why: the postcondition assertion (`mean ≈ 0`) restates the task's
*definition* — a cheap self-check, and `allclose` (not `==`) because float
arithmetic.


<!-- dd:dd-q15 -->

### Problem 15 · faded — your turn

Subtract the global mean (any input shape).

**Expected output** — run the cell below once `solve` is right and it should print this.

```text
tensor([[-2., -1.],
        [ 0.,  3.]])
```


In [ ]:
import torch as t

def solve(z):
    """z minus its overall mean; result's mean is 0."""
    return z - z._____()


# Example run — the grader calls solve() with several arrays.
example = t.tensor([[1.0, 2.0], [3.0, 6.0]])
print(solve(example))


In [ ]:
# Did it work? Run this. (NameError → run the checker cell at
# the top of the notebook first: Runtime ▸ Run before.)
dd_check(15)


In [ ]:
#@title 💡 Solution — Problem 15
# Running this rebinds `solve` to the reference answer. Re-run your
# own cell before dd_check() again, or you are checking this one.
import torch as t

def solve(z):
    return z - z.mean()


example = t.tensor([[1.0, 2.0], [3.0, 6.0]])
print(solve(example))


<!-- dd:dd-seg-numpy-centering-1 -->

### row and column centering — where keepdims earns its keep


Per-group centering picks an axis: which group shares a statistic?
Reading a task, the phrase "each row's …" means axis=1; "each column's …"
means axis=0.

- **Row centering**: `z - z.mean(axis=1, keepdims=True)` — the (r, 1)
  statistic broadcasts across each row; every row of the result averages 0.
- **Column centering**: `z - z.mean(axis=0)` — the (c,) statistic
  right-aligns against z's last axis, hitting each column.

Note the asymmetry: columns work WITHOUT keepdims because right-alignment
happens to be correct; rows NEED keepdims (or `[:, None]`). When in doubt,
keepdims is never wrong.


In [ ]:
import torch as t

z = t.tensor([[1.0, 2.0, 3.0],
              [10.0, 20.0, 30.0]])

# ROW centering: statistic per row -> dim=1, kept as a column (2, 1)
# so it broadcasts back across each row.
row_mu = z.mean(dim=1, keepdim=True)
assert row_mu.tolist() == [[2.0], [20.0]]
centered = z - row_mu
assert centered.tolist() == [[-1.0, 0.0, 1.0],
                             [-10.0, 0.0, 10.0]]
# Postcondition worth checking in real code: every row now averages 0.
assert t.allclose(centered.mean(dim=1), t.zeros(2))
print("row means, kept as a column", tuple(row_mu.shape), ":", row_mu.tolist())
print(centered)
print("each row now averages", centered.mean(dim=1).tolist())




Why: materializing `row_mu` and asserting its SHAPE (2, 1) before
subtracting is the discipline that prevents the classic square-matrix bug
(bare (2,) statistic aligning against the wrong axis).


<!-- dd:dd-q7 -->

### Problem 7 · faded — your turn

Subtract each row's own mean.

**Expected output** — run the cell below once `solve` is right and it should print this.

```text
tensor([[ -1.,   0.,   1.],
        [-10.,   0.,  10.]])
```


In [ ]:
import torch as t

def solve(z):
    """Every entry minus its row's mean; each result row averages 0."""
    return z - z.mean(_____=_____, keepdim=_____)


# Example run — the grader calls solve() with several matrices.
example = t.tensor([[1.0, 2.0, 3.0], [10.0, 20.0, 30.0]])
print(solve(example))


In [ ]:
# Did it work? Run this. (NameError → run the checker cell at
# the top of the notebook first: Runtime ▸ Run before.)
dd_check(7)


In [ ]:
#@title 💡 Solution — Problem 7
# Running this rebinds `solve` to the reference answer. Re-run your
# own cell before dd_check() again, or you are checking this one.
import torch as t

def solve(z):
    return z - z.mean(dim=1, keepdim=True)


example = t.tensor([[1.0, 2.0, 3.0], [10.0, 20.0, 30.0]])
print(solve(example))


<!-- dd:dd-seg-numpy-centering-2 -->

### standardizing — apply the template twice


**Standardizing** (z-scores) is centering AND scaling:
`(x - x.mean(axis=0)) / x.std(axis=0)` gives each column mean 0, std 1.
Both reductions share the same axis — mixing axes between the mean and the
std is a real bug seen in the wild; the template keeps them locked together.

One refinement for real data: when the divisor can be zero (a constant
column has std 0), guard the division with the `where=`/`out=` pattern from
the where-select KP — statistics and safe division compose cleanly.


In [ ]:
import torch as t

# COLUMN standardizing: both statistics per column (dim=0).
x = t.tensor([[1.0, 10.0],
              [3.0, 30.0]])
zscores = (x - x.mean(dim=0)) / x.std(dim=0, correction=0)
assert t.allclose(zscores, t.tensor([[-1.0, -1.0],
                                     [1.0, 1.0]]))
assert t.allclose(zscores.mean(dim=0), t.zeros(2))
assert t.allclose(zscores.std(dim=0, correction=0), t.ones(2))
print("column means", x.mean(dim=0), "| column sds",
      x.std(dim=0, correction=0))
print(zscores)
print("after: means", zscores.mean(dim=0), "sds",
      zscores.std(dim=0, correction=0))




Why: the two postconditions (`mean ≈ 0`, `std ≈ 1`) catch dim mistakes
instantly. Note the `correction=0`: PyTorch's `std` defaults to the SAMPLE
standard deviation (divide by n−1), so the population version these drills
want has to be asked for by name. Leaving it off does not raise — it just
returns slightly different numbers, which is the worst way to be wrong.


<!-- dd:dd-q106 -->

### Problem 106 · faded — your turn

Standardize each column (no constant columns).

**Expected output** — run the cell below once `solve` is right and it should print this.

```text
tensor([[-1., -1.],
        [ 1.,  1.]])
```


In [ ]:
import torch as t

def solve(x):
    """Each column standardized: mean 0, std 1."""
    return (x - x.mean(_____=_____)) / x._____(_____=_____, _____=0)


# Example run — the grader calls solve() with several matrices.
example = t.tensor([[1.0, 10.0], [3.0, 30.0]])
print(solve(example))


In [ ]:
# Did it work? Run this. (NameError → run the checker cell at
# the top of the notebook first: Runtime ▸ Run before.)
dd_check(106)


In [ ]:
#@title 💡 Solution — Problem 106
# Running this rebinds `solve` to the reference answer. Re-run your
# own cell before dd_check() again, or you are checking this one.
import torch as t

def solve(x):
    return (x - x.mean(dim=0)) / x.std(dim=0, correction=0)


example = t.tensor([[1.0, 10.0], [3.0, 30.0]])
print(solve(example))


<!-- dd:dd-q154 -->

### Problem 154 · guided

Write a function solve(z, limit) that takes a 2-D float tensor (no constant columns) and a positive float limit. First standardize each COLUMN (subtract its mean, divide by its population std), then clip the standardized values into [-limit, +limit], and return the result.

**Expected output** — run the cell below once `solve` is right and it should print this.

```text
tensor([[-0.7440, -1.0000],
        [-0.6696,  0.0000],
        [ 1.0000,  1.0000]])
```


<details>
<summary>Hints</summary>

1. Two steps, in order: standardize per COLUMN, then clip. Column
   statistics reduce over dim=0.
2. Population std means dividing by n, not n-1 — torch's default is the
   sample version, so one keyword has to change.
3. `(z - z.mean(dim=0)) / z.std(dim=0, correction=0)`, then `t.clip(...,
   -limit, limit)`.

</details>


In [ ]:
import torch as t

def solve(z, limit):
    """Return column-standardized z clipped to [-limit, limit]."""
    return None


# Example run — the grader calls solve() with several inputs.
example = t.tensor([[1.0, 10.0], [2.0, 20.0], [30.0, 30.0]])
print(solve(example, 1.0))


In [ ]:
# Did it work? Run this. (NameError → run the checker cell at
# the top of the notebook first: Runtime ▸ Run before.)
dd_check(154)


In [ ]:
#@title 💡 Solution — Problem 154
# Running this rebinds `solve` to the reference answer. Re-run your
# own cell before dd_check() again, or you are checking this one.
import torch as t

def solve(z, limit):
    standardized = (z - z.mean(dim=0)) / z.std(dim=0, correction=0)
    return t.clip(standardized, -limit, limit)


example = t.tensor([[1.0, 10.0], [2.0, 20.0], [30.0, 30.0]])
print(solve(example, 1.0))


<!-- dd:dd-q109 -->

### Problem 109 · independent

Write a function solve(x) that takes a 2-D PyTorch float tensor and returns a new tensor of the same shape in which every entry has had its own COLUMN's mean subtracted, so each column of the result averages to zero. (Contrast: an earlier drill centers each row.) Do not modify the input.

**Expected output** — run the cell below once `solve` is right and it should print this.

```text
tensor([[-1., -5.],
        [ 1.,  5.]])
```


In [ ]:
import torch as t

def solve(x):
    """Return x with each column centered on its own mean."""
    return None


# Example run — the grader calls solve() with several matrices.
example = t.tensor([[1.0, 10.0], [3.0, 20.0]])
print(solve(example))


In [ ]:
# Did it work? Run this. (NameError → run the checker cell at
# the top of the notebook first: Runtime ▸ Run before.)
dd_check(109)


In [ ]:
#@title 💡 Solution — Problem 109
# Running this rebinds `solve` to the reference answer. Re-run your
# own cell before dd_check() again, or you are checking this one.
import torch as t

def solve(x):
    return x - x.mean(dim=0)


example = t.tensor([[1.0, 10.0], [3.0, 20.0]])
print(solve(example))


<!-- dd:dd-q218 -->

### Problem 218 · independent

Write a function solve(z) that takes a 2-D float tensor and standardizes each COLUMN (subtract its mean, divide by its population std) — but columns whose std is ZERO (constant columns) must come out as all zeros (just centered), with no division warnings or NaN. (An earlier drill assumes no constant columns; handling them is this one's point.)

**Expected output** — run the cell below once `solve` is right and it should print this.

```text
tensor([[-1.,  0.],
        [ 1.,  0.]])
```


In [ ]:
import torch as t

def solve(z):
    """Column-standardize z; constant columns become all zeros."""
    return None


# Example run — the grader calls solve() with several arrays.
print(solve(t.tensor([[1.0, 5.0], [3.0, 5.0]])))


In [ ]:
# Did it work? Run this. (NameError → run the checker cell at
# the top of the notebook first: Runtime ▸ Run before.)
dd_check(218)


In [ ]:
#@title 💡 Solution — Problem 218
# Running this rebinds `solve` to the reference answer. Re-run your
# own cell before dd_check() again, or you are checking this one.
import torch as t

def solve(z):
    mu = z.mean(dim=0, keepdim=True)
    sd = z.std(dim=0, keepdim=True, correction=0)
    return (z - mu) / t.where(sd == 0, 1, sd)


print(solve(t.tensor([[1.0, 5.0], [3.0, 5.0]])))


#### Common mistakes

- **"`z - z.mean(dim=1)` centers the rows."** — On a non-square matrix it
  ERRORS; on a square one it silently centers the wrong way (the bare (r,)
  aligns with columns). Row statistics need `keepdim=True` (or `[:, None]`).
- **"`std()` divides by n."** — PyTorch divides by n−1 by default
  (`correction=1`, the SAMPLE std). The population version is
  `std(correction=0)`. This is the opposite of NumPy's default, so a formula
  carried over from the numpy dialect changes its answer without complaining
  — always state `correction` explicitly.
- **"Center-then-scale needs a loop over rows."** — The statistic/broadcast
  template does any per-row/per-column normalization in one expression;
  loops over rows are a sign the axis machinery isn't being used.


<!-- dd:dd-kp-numpy-rescaling -->

## Rescaling — min-max, unit norm, probability rows

`numpy.rescaling`


Centering shifts data; **rescaling** maps it into a target range or size.
Three canonical scalings cover nearly every drill, each defined by what the
result must satisfy:

- **Min-max to [0, 1]**: `(z - z.min()) / (z.max() - z.min())` — smallest
  entry becomes exactly 0, largest exactly 1, everything else keeps its
  relative position. ("Normalize to [a, b]" is this, then `* (b - a) + a`.)
- **Unit Euclidean length**: `v / t.linalg.norm(v)` — same direction, length
  1. Scaling to length L is `v * (L / norm)`. `t.linalg.norm` is the
  square-root-of-sum-of-squares; on matrices it takes `axis=` (and
  `keepdims=`) exactly like a reduction.
- **Probability distribution**: `z / z.sum()` — non-negative entries summing
  to 1. Per-row: divide by `z.sum(axis=1, keepdims=True)`.

All three are the same template as centering — *statistic, then divide* —
so per-row/per-column versions come from `axis=` + `keepdims=True`, no new
machinery.

The recurring production concern: **the divisor can be zero** (all-zero row,
constant array). Bare division emits warnings and produces NaN/Inf; drills
phrased "zero rows must remain zeros, with no warnings" want the safe-divide
composition:

```python no-run
sums = z.sum(dim=1, keepdim=True)
out = t.zeros_like(z)
nz = (sums != 0).squeeze(1)      # which ROWS are safe to divide
out[nz] = z[nz] / sums[nz]
```

— the row mask keeps the bad rows out of the division entirely, and the
zeros canvas is what they keep. (`.squeeze(1)` turns the (r, 1) column of
flags into the (r,) row mask that indexing wants.)


Task: min-max a vector to [0, 1]; make a unit vector; convert score rows to
probability rows, zero rows staying zero.


In [ ]:
import torch as t

# 1. Min-max: affine map sending min -> 0 and max -> 1.
z = t.tensor([2.0, 4.0, 6.0])
mm = (z - z.min()) / (z.max() - z.min())
assert mm.tolist() == [0.0, 0.5, 1.0]

# 2. Unit length: divide by the Euclidean norm. Direction is preserved —
# the entries keep their ratios (3:4 here).
v = t.tensor([3.0, 4.0])
unit = v / t.linalg.norm(v)
assert t.allclose(unit, t.tensor([0.6, 0.8]))
assert t.isclose(t.linalg.norm(unit), t.tensor(1.0))   # defining postcondition

# 3. Probability rows with a zero row — the safe-divide composition.
scores = t.tensor([[1.0, 3.0],
                   [0.0, 0.0]])
sums = scores.sum(dim=1, keepdim=True)          # (2, 1): [[4.], [0.]]
probs = t.zeros_like(scores)
nz = (sums != 0).squeeze(1)                     # row 0 only
probs[nz] = scores[nz] / sums[nz]
assert probs.tolist() == [[0.25, 0.75],
                          [0.0, 0.0]]
assert t.isclose(probs[0].sum(), t.tensor(1.0))
print("min-max  ", z, "->", mm)
print("unit     ", v, "->", unit, "norm", t.linalg.norm(unit).item())
print("row sums ", sums.squeeze(1), "-> divisible rows:", nz)
print(probs, " <- the all-zero row stayed zero instead of becoming nan")




Why each step:

1. Each scaling is verified against its own DEFINITION (endpoints 0 and 1;
   norm 1; row sums 1) — write these postconditions as asserts while
   practicing and axis/formula errors have nowhere to hide.
2. In the min-max formula, both statistics come from the SAME array before
   any modification — compute them first (or inline), never after a partial
   in-place update.
3. Step 3 is three prior KPs snapping together: axis reduction (row sums),
   keepdims (alignment), where/out (safety). Recognizing tasks as
   compositions of known moves — rather than new tricks — is the skill this
   lesson is building.


<!-- dd:dd-q80 -->

### Problem 80 · faded — your turn

Linear rescale so min → 0.0 and max → 1.0.

**Expected output** — run the cell below once `solve` is right and it should print this.

```text
tensor([0.0000, 0.5000, 1.0000])
```


In [ ]:
import torch as t

def solve(z):
    """Min-max rescale to [0, 1]."""
    return (z - _____) / (_____ - _____)


# Example run — the grader calls solve() with several vectors.
example = t.tensor([2.0, 4.0, 6.0])
print(solve(example))


In [ ]:
# Did it work? Run this. (NameError → run the checker cell at
# the top of the notebook first: Runtime ▸ Run before.)
dd_check(80)


In [ ]:
#@title 💡 Solution — Problem 80
# Running this rebinds `solve` to the reference answer. Re-run your
# own cell before dd_check() again, or you are checking this one.
import torch as t

def solve(z):
    return (z - z.min()) / (z.max() - z.min())


example = t.tensor([2.0, 4.0, 6.0])
print(solve(example))


Task: min-max a vector to [0, 1]; make a unit vector; convert score rows to
probability rows, zero rows staying zero.


In [ ]:
import torch as t

# 1. Min-max: affine map sending min -> 0 and max -> 1.
z = t.tensor([2.0, 4.0, 6.0])
mm = (z - z.min()) / (z.max() - z.min())
assert mm.tolist() == [0.0, 0.5, 1.0]

# 2. Unit length: divide by the Euclidean norm. Direction is preserved —
# the entries keep their ratios (3:4 here).
v = t.tensor([3.0, 4.0])
unit = v / t.linalg.norm(v)
assert t.allclose(unit, t.tensor([0.6, 0.8]))
assert t.isclose(t.linalg.norm(unit), t.tensor(1.0))   # defining postcondition

# 3. Probability rows with a zero row — the safe-divide composition.
scores = t.tensor([[1.0, 3.0],
                   [0.0, 0.0]])
sums = scores.sum(dim=1, keepdim=True)          # (2, 1): [[4.], [0.]]
probs = t.zeros_like(scores)
nz = (sums != 0).squeeze(1)                     # row 0 only
probs[nz] = scores[nz] / sums[nz]
assert probs.tolist() == [[0.25, 0.75],
                          [0.0, 0.0]]
assert t.isclose(probs[0].sum(), t.tensor(1.0))
print("min-max  ", z, "->", mm)
print("unit     ", v, "->", unit, "norm", t.linalg.norm(unit).item())
print("row sums ", sums.squeeze(1), "-> divisible rows:", nz)
print(probs, " <- the all-zero row stayed zero instead of becoming nan")




Why each step:

1. Each scaling is verified against its own DEFINITION (endpoints 0 and 1;
   norm 1; row sums 1) — write these postconditions as asserts while
   practicing and axis/formula errors have nowhere to hide.
2. In the min-max formula, both statistics come from the SAME array before
   any modification — compute them first (or inline), never after a partial
   in-place update.
3. Step 3 is three prior KPs snapping together: axis reduction (row sums),
   keepdims (alignment), where/out (safety). Recognizing tasks as
   compositions of known moves — rather than new tricks — is the skill this
   lesson is building.


<!-- dd:dd-q97 -->

### Problem 97 · faded — your turn

Same template, one step further out. The example divides by the norm, which
pins the length at exactly 1. Here the target length is an argument — so the
scale factor is no longer "the norm", it is whatever sends the norm to
`length`. Write that factor; the direction takes care of itself.

**Expected output** — run the cell below once `solve` is right and it should print this.

```text
tensor([6., 8.])
```


In [ ]:
import torch as t

def solve(v, length):
    """v rescaled so its Euclidean norm is exactly `length`, direction kept."""
    return v * (_____ / t.linalg.norm(v))


# Example run — the grader calls solve() with several inputs.
print(solve(t.tensor([3.0, 4.0]), 10.0))


In [ ]:
# Did it work? Run this. (NameError → run the checker cell at
# the top of the notebook first: Runtime ▸ Run before.)
dd_check(97)


In [ ]:
#@title 💡 Solution — Problem 97
# Running this rebinds `solve` to the reference answer. Re-run your
# own cell before dd_check() again, or you are checking this one.
import torch as t

def solve(v, length):
    return v * (length / t.linalg.norm(v))


print(solve(t.tensor([3.0, 4.0]), 10.0))


<!-- dd:dd-q162 -->

### Problem 162 · guided

Write a function solve(z) that takes a 2-D float tensor with non-negative entries and returns the tensor with each ROW rescaled to sum to 1 — a probability distribution per row. Rows that sum to zero must remain all zeros, with no warnings, NaN, or inf produced.

**Expected output** — run the cell below once `solve` is right and it should print this.

```text
tensor([[0.2500, 0.7500],
        [0.0000, 0.0000]])
```


<details>
<summary>Hints</summary>

1. Each row becomes a probability distribution — divide by what, per row,
   kept in which shape?
2. Zero rows must SURVIVE as zeros — that's the masked division, not an
   if-statement.
3. `out = t.zeros_like(z)`, `nz = (sums != 0).squeeze(1)`,
   `out[nz] = z[nz] / sums[nz]`, with `sums = z.sum(dim=1, keepdim=True)`.

</details>


In [ ]:
import torch as t

def solve(z):
    """Return z with each row normalized to sum 1; zero rows stay zero."""
    return None


# Example run — the grader calls solve() with several arrays.
print(solve(t.tensor([[1.0, 3.0], [0.0, 0.0]])))


In [ ]:
# Did it work? Run this. (NameError → run the checker cell at
# the top of the notebook first: Runtime ▸ Run before.)
dd_check(162)


In [ ]:
#@title 💡 Solution — Problem 162
# Running this rebinds `solve` to the reference answer. Re-run your
# own cell before dd_check() again, or you are checking this one.
import torch as t
def solve(z):
    sums = z.sum(dim=1, keepdim=True)
    out = t.zeros_like(z)
    nz = (sums != 0).squeeze(1)
    out[nz] = z[nz] / sums[nz]
    return out


print(solve(t.tensor([[1.0, 3.0], [0.0, 0.0]])))


<!-- dd:dd-q6 -->

### Problem 6 · independent

Write a function solve(v) that takes a 1-D PyTorch tensor of floats with nonzero length (and at least one nonzero entry) and returns the vector rescaled to unit length: the result points in the same direction but its Euclidean length equals 1. Do not modify the input tensor.

**Expected output** — run the cell below once `solve` is right and it should print this.

```text
tensor([0.6000, 0.8000])
```


In [ ]:
import torch as t

def solve(v):
    """Return v scaled so its Euclidean length is exactly 1."""
    return None


# Example run — the grader calls solve() with several vectors.
example = t.tensor([3.0, 4.0])
print(solve(example))


In [ ]:
# Did it work? Run this. (NameError → run the checker cell at
# the top of the notebook first: Runtime ▸ Run before.)
dd_check(6)


In [ ]:
#@title 💡 Solution — Problem 6
# Running this rebinds `solve` to the reference answer. Re-run your
# own cell before dd_check() again, or you are checking this one.
import torch as t

def solve(v):
    return v / t.linalg.norm(v)


example = t.tensor([3.0, 4.0])
print(solve(example))


<!-- dd:dd-q153 -->

### Problem 153 · independent

Write a function solve(z) that takes a 2-D float tensor and returns the tensor with each ROW rescaled to unit Euclidean length — except rows that are entirely zero, which must remain all zeros (no NaN or inf anywhere, and no warnings raised).

**Expected output** — run the cell below once `solve` is right and it should print this.

```text
tensor([[0.6000, 0.8000],
        [0.0000, 0.0000]])
```


In [ ]:
import torch as t

def solve(z):
    """Return z with each row scaled to unit length; zero rows stay zero."""
    return None


# Example run — the grader calls solve() with several arrays.
print(solve(t.tensor([[3.0, 4.0], [0.0, 0.0]])))


In [ ]:
# Did it work? Run this. (NameError → run the checker cell at
# the top of the notebook first: Runtime ▸ Run before.)
dd_check(153)


In [ ]:
#@title 💡 Solution — Problem 153
# Running this rebinds `solve` to the reference answer. Re-run your
# own cell before dd_check() again, or you are checking this one.
import torch as t
def solve(z):
    norms = t.linalg.norm(z, dim=1, keepdim=True)
    out = t.zeros_like(z)
    nz = (norms != 0).squeeze(1)
    out[nz] = z[nz] / norms[nz]
    return out


print(solve(t.tensor([[3.0, 4.0], [0.0, 0.0]])))


<!-- dd:dd-q10 -->

### Problem 10 · independent

Write a function solve(z) that takes a PyTorch tensor of floats (any shape, not all entries equal) and returns a new tensor of the same shape rescaled so that its overall mean is 0 and its overall standard deviation is 1. Use the population standard deviation (PyTorch's default). Do not modify the input.

**Expected output** — run the cell below once `solve` is right and it should print this.

```text
tensor([[-1.3416, -0.4472],
        [ 0.4472,  1.3416]])
```


In [ ]:
import torch as t

def solve(z):
    """Return z standardized: overall mean 0, overall std 1."""
    return None


# Example run — the grader calls solve() with several arrays.
example = t.tensor([[1.0, 2.0], [3.0, 4.0]])
print(solve(example))


In [ ]:
# Did it work? Run this. (NameError → run the checker cell at
# the top of the notebook first: Runtime ▸ Run before.)
dd_check(10)


In [ ]:
#@title 💡 Solution — Problem 10
# Running this rebinds `solve` to the reference answer. Re-run your
# own cell before dd_check() again, or you are checking this one.
import torch as t

def solve(z):
    return (z - z.mean()) / z.std(correction=0)


example = t.tensor([[1.0, 2.0], [3.0, 4.0]])
print(solve(example))


<!-- dd:dd-q103 -->

### Problem 103 · independent

Write a function solve(z) that takes a 1-D PyTorch float tensor of logits and returns its softmax: a same-shape tensor of non-negative entries summing to 1, where entry i is exp(z[i]) divided by the sum of exponentials. Your implementation must be numerically stable — it will be tested with large logits like 1000 where naive exponentiation overflows to inf.

**Expected output** — run the cell below once `solve` is right and it should print this.

```text
tensor([0.0900, 0.2447, 0.6652])
```


In [ ]:
import torch as t

def solve(z):
    """Return the softmax of the logit vector z (numerically stable)."""
    return None


# Example run — the grader calls solve() with several logit vectors.
print(solve(t.tensor([1.0, 2.0, 3.0])))


In [ ]:
# Did it work? Run this. (NameError → run the checker cell at
# the top of the notebook first: Runtime ▸ Run before.)
dd_check(103)


In [ ]:
#@title 💡 Solution — Problem 103
# Running this rebinds `solve` to the reference answer. Re-run your
# own cell before dd_check() again, or you are checking this one.
import torch as t

def solve(z):
    e = t.exp(z - z.max())
    return e / e.sum()


print(solve(t.tensor([1.0, 2.0, 3.0])))


<!-- dd:dd-q147 -->

### Problem 147 · independent

Write a function solve(z, lo_pct, hi_pct) that takes a 1-D float tensor and two percentile levels (e.g. 5 and 95), and returns the WINSORIZED tensor: values below the lo_pct percentile are raised to it, values above the hi_pct percentile are lowered to it, everything else is unchanged. Do not modify the input.

**Expected output** — run the cell below once `solve` is right and it should print this.

```text
tensor([ 0.4000,  1.0000,  2.0000,  3.0000, 61.2000])
```


In [ ]:
import torch as t

def solve(z, lo_pct, hi_pct):
    """Return z clipped to its [lo_pct, hi_pct] percentile range."""
    return None


# Example run — the grader calls solve() with several inputs.
print(solve(t.tensor([0.0, 1.0, 2.0, 3.0, 100.0]), 10, 90))


In [ ]:
# Did it work? Run this. (NameError → run the checker cell at
# the top of the notebook first: Runtime ▸ Run before.)
dd_check(147)


In [ ]:
#@title 💡 Solution — Problem 147
# Running this rebinds `solve` to the reference answer. Re-run your
# own cell before dd_check() again, or you are checking this one.
import torch as t
def solve(z, lo_pct, hi_pct):
    qs = t.tensor([lo_pct / 100, hi_pct / 100], dtype=z.dtype)
    lo, hi = t.quantile(z, qs)
    return t.clip(z, lo, hi)


print(solve(t.tensor([0.0, 1.0, 2.0, 3.0, 100.0]), 10, 90))


#### Common mistakes

- **"Normalize means divide by the max."** — `z / z.max()` sends the max to 1
  but the min to min/max, not 0. True min-max subtracts the min first. Read
  which endpoints the task pins down.
- **"norm(v) is the sum of absolute values."** — Default is the EUCLIDEAN
  (L2) norm: √Σx². The L1 norm is `t.linalg.norm(v, 1)` — and a
  "probability" scaling divides by the plain SUM, which for non-negative
  data equals L1.
- **"Guard zero divisors with `if z.sum() == 0`."** — Per-row that's a loop
  in disguise. The vectorized guard is a zeros canvas plus a row mask, which
  handles mixed zero/nonzero rows without an if-statement. (NumPy spells this
  `np.divide(..., out=..., where=...)`; PyTorch has no `where=` keyword.)


<!-- dd:dd-kp-numpy-cumulative-diff -->

## Cumulative ops and discrete differences

`numpy.cumulative-diff`


<!-- dd:dd-seg-numpy-cumulative-diff-0 -->

### cumsum — running totals


Take a 1-D tensor `x` — one row of numbers, like five days of sales.
`t.cumsum(x, dim=0)` returns its **running total**: entry i is
`x[0] + … + x[i]`. The result is the same length as `x`, and its last entry is
the same number `x.sum()` would give you. It is the vectorized replacement for
the "total so far" loop.

`dim=0` says which direction to add along. A 1-D tensor has only one direction,
so `dim=0` is the only choice here — but torch makes you say it. NumPy would
have flattened the tensor and guessed; torch never guesses.

Reading a task: "running / so far / cumulative" is the tell for this family.


In [ ]:
import torch as t

# Five days of sales, as a 1-D tensor — one number per day.
sales = t.tensor([2, 3, 5, 1, 4])
print(sales)

# The running total. Entry i is the sum of days 0 through i, so entry 2 is
# 2 + 3 + 5 = 10. Same length as sales: one running total per day.
totals = t.cumsum(sales, dim=0)
print(totals)

# The LAST running total is the total of everything...
print(totals[-1])

# ...which is exactly what sum() gives you in one step.
print(sales.sum())




```output
tensor([2, 3, 5, 1, 4])
tensor([ 2,  5, 10, 11, 15])
tensor(15)
tensor(15)
```

Why: those last two prints are the anchor. `sum` gives you the final answer;
`cumsum` gives you every partial answer along the way, and its last entry is
the final one. Same computation, different amount of it kept.


<!-- dd:dd-q234 -->

### Problem 234 · faded — your turn

Running total of a 1-D tensor.

**Expected output** — run the cell below once `solve` is right and it should print this.

```text
tensor([ 2,  5, 10])
```


In [ ]:
import torch as t

def solve(x):
    """Entry i = total of x[0..i] inclusive."""
    return t._____(x, _____=0)


# Example run — the grader calls solve() with several different arrays,
# including edge cases. Your function must work for all of them.
example = t.tensor([2, 3, 5])
print(solve(example))


In [ ]:
# Did it work? Run this. (NameError → run the checker cell at
# the top of the notebook first: Runtime ▸ Run before.)
dd_check(234)


In [ ]:
#@title 💡 Solution — Problem 234
# Running this rebinds `solve` to the reference answer. Re-run your
# own cell before dd_check() again, or you are checking this one.
import torch as t
def solve(x):
    return t.cumsum(x, dim=0)


example = t.tensor([2, 3, 5])
print(solve(example))


<!-- dd:dd-seg-numpy-cumulative-diff-1 -->

### the cum* family — running ANYTHING


cumsum has siblings, and they all mean "so far". `t.cummax(x, dim)` is the
running **maximum** — the largest value seen up to that point. `t.cummin` is
the running minimum, and `t.cumprod` the running product.

The two extrema return a *pair*, not a single tensor: `.values` holds the
running values and `.indices` tells you where each new record was set. That is
the same shape `t.sort` and `t.topk` return, so the `.values` step will keep
coming up.

Every one of them requires an explicit `dim`, exactly like cumsum.


In [ ]:
import torch as t

# The same five days of sales.
sales = t.tensor([2, 3, 5, 1, 4])
print(sales)

# cummax returns a PAIR, so this is not the tensor you want yet.
record_pair = t.cummax(sales, dim=0)
print(record_pair)

# .values pulls out the running maximum: the best day so far. It never goes
# down — once you have seen a 5, the best-so-far stays at least 5.
records = record_pair.values
print(records)

# .indices says WHICH day set each record. Days 3 and 4 did not beat day 2,
# so both still point back at index 2.
print(record_pair.indices)




```output
tensor([2, 3, 5, 1, 4])
torch.return_types.cummax(
values=tensor([2, 3, 5, 5, 5]),
indices=tensor([0, 1, 2, 2, 2]))
tensor([2, 3, 5, 5, 5])
tensor([0, 1, 2, 2, 2])
```

Why: `sales.amax()` collapses the whole tensor to one number, 5. The running
version keeps the answer at every point instead. "So far" in a task is the
tell that you want a `cum*` and not a plain reduction.


<!-- dd:dd-q152 -->

### Problem 152 · faded — your turn

Running maximum of each row, scanning left to right.

**Expected output** — run the cell below once `solve` is right and it should print this.

```text
tensor([[3, 3, 4, 4, 5]])
```


In [ ]:
import torch as t

def solve(z):
    """Entry [i, j] = max of row i's columns 0..j."""
    return t._____(z, _____=1).values


# Example run — the grader calls solve() with several arrays.
print(solve(t.tensor([[3, 1, 4, 1, 5]])))


In [ ]:
# Did it work? Run this. (NameError → run the checker cell at
# the top of the notebook first: Runtime ▸ Run before.)
dd_check(152)


In [ ]:
#@title 💡 Solution — Problem 152
# Running this rebinds `solve` to the reference answer. Re-run your
# own cell before dd_check() again, or you are checking this one.
import torch as t
def solve(z):
    return t.cummax(z, dim=1).values


print(solve(t.tensor([[3, 1, 4, 1, 5]])))


<!-- dd:dd-seg-numpy-cumulative-diff-2 -->

### t.diff — adjacent differences


`t.diff(x)` gives the **step between neighbours**: entry i is
`x[i+1] - x[i]`. The result is one shorter than the input, because n numbers
have only n−1 gaps between them.

It is the opposite of cumsum. cumsum adds a sequence up; diff reads back the
individual steps. `n=k` applies diff k times, and on a matrix `dim=` picks the
direction, the same argument cumsum takes.

Reading a task: "successive / adjacent / change between neighbours" → diff.


In [ ]:
import torch as t

# The same five days of sales.
sales = t.tensor([2, 3, 5, 1, 4])
print(sales)

# The day-to-day change. 2 -> 3 is +1, 3 -> 5 is +2, 5 -> 1 is -4, 1 -> 4 is
# +3. Four gaps between five days, so the result is one shorter.
changes = t.diff(sales)
print(changes)

# diff undoes cumsum: the steps of a running total ARE the original numbers,
# starting from the second one.
print(t.diff(t.cumsum(sales, dim=0)))

# ...which is sales without its first entry.
print(sales[1:])




```output
tensor([2, 3, 5, 1, 4])
tensor([ 1,  2, -4,  3])
tensor([3, 5, 1, 4])
tensor([3, 5, 1, 4])
```

Why: the round trip is the point — cumsum and diff are inverses, and the
length bookkeeping (n going to n−1) is the off-by-one this family is famous
for.


<!-- dd:dd-q22 -->

### Problem 22 · faded — your turn

Differences between adjacent columns within each row: result (r, c-1).

**Expected output** — run the cell below once `solve` is right and it should print this.

```text
tensor([[1, 1],
        [1, 1],
        [1, 1]])
```


In [ ]:
import torch as t

def solve(z):
    """Entry [i, j] = z[i, j+1] - z[i, j]."""
    return t._____(z, _____=_____)


# Example run — the grader calls solve() with several matrices.
example = t.arange(9).reshape(3, 3)
print(solve(example))


In [ ]:
# Did it work? Run this. (NameError → run the checker cell at
# the top of the notebook first: Runtime ▸ Run before.)
dd_check(22)


In [ ]:
#@title 💡 Solution — Problem 22
# Running this rebinds `solve` to the reference answer. Re-run your
# own cell before dd_check() again, or you are checking this one.
import torch as t

def solve(z):
    return t.diff(z, dim=1)


example = t.arange(9).reshape(3, 3)
print(solve(example))


<!-- dd:dd-q68 -->

### Problem 68 · guided

Write a function solve(z) that takes a 1-D PyTorch numeric tensor and returns a plain Python bool: True when z is non-decreasing (every element is >= the one before it), False otherwise. An tensor with a single element counts as non-decreasing.

**Expected output** — run the cell below once `solve` is right and it should print this.

```text
True
```


<details>
<summary>Hints</summary>

1. Non-decreasing is a statement about consecutive PAIRS, and the pairwise
   gaps are exactly what one function gives you.
2. Once you have the gaps, the whole question is whether all of them are
   >= 0.
3. `bool(t.all(t.diff(z) >= 0))` — the `bool()` matters, the drill wants a
   Python bool. A length-1 tensor gives an empty diff, and `all()` of
   nothing is True, which is the answer you want.

</details>


In [ ]:
import torch as t

def solve(z):
    """Return True if z is non-decreasing, else False."""
    return None


# Example run — the grader calls solve() with several arrays.
example = t.tensor([1, 2, 2, 4, 5])
print(solve(example))


In [ ]:
# Did it work? Run this. (NameError → run the checker cell at
# the top of the notebook first: Runtime ▸ Run before.)
dd_check(68)


In [ ]:
#@title 💡 Solution — Problem 68
# Running this rebinds `solve` to the reference answer. Re-run your
# own cell before dd_check() again, or you are checking this one.
import torch as t

def solve(z):
    return bool(t.all(t.diff(z) >= 0))


example = t.tensor([1, 2, 2, 4, 5])
print(solve(example))


<!-- dd:dd-q82 -->

### Problem 82 · independent

Write a function solve(z) that takes a 2-D PyTorch integer tensor and returns an tensor of the same shape in which entry [i, j] equals the running total of row i from column 0 through column j inclusive (a cumulative sum along each row).

**Expected output** — run the cell below once `solve` is right and it should print this.

```text
tensor([[ 0,  1,  3],
        [ 3,  7, 12]])
```


In [ ]:
import torch as t

def solve(z):
    """Return the cumulative sum of z along each row."""
    return None


# Example run — the grader calls solve() with several matrices.
example = t.arange(6).reshape(2, 3)
print(solve(example))


In [ ]:
# Did it work? Run this. (NameError → run the checker cell at
# the top of the notebook first: Runtime ▸ Run before.)
dd_check(82)


In [ ]:
#@title 💡 Solution — Problem 82
# Running this rebinds `solve` to the reference answer. Re-run your
# own cell before dd_check() again, or you are checking this one.
import torch as t

def solve(z):
    return z.cumsum(dim=1)


example = t.arange(6).reshape(2, 3)
print(solve(example))


<!-- dd:dd-q149 -->

### Problem 149 · independent

Write a function solve(a, k) that takes a 1-D PyTorch integer tensor and a positive integer k, and returns the k-th order discrete difference of a: applying the adjacent-difference operation k times in succession. The result has length len(a) - k.

**Expected output** — run the cell below once `solve` is right and it should print this.

```text
tensor([2, 2, 2])
```


In [ ]:
import torch as t

def solve(a, k):
    """Return the k-th order discrete difference of a."""
    return None


# Example run — the grader calls solve() with several inputs.
print(solve(t.tensor([1, 4, 9, 16, 25]), 2))


In [ ]:
# Did it work? Run this. (NameError → run the checker cell at
# the top of the notebook first: Runtime ▸ Run before.)
dd_check(149)


In [ ]:
#@title 💡 Solution — Problem 149
# Running this rebinds `solve` to the reference answer. Re-run your
# own cell before dd_check() again, or you are checking this one.
import torch as t

def solve(a, k):
    return t.diff(a, n=k)


print(solve(t.tensor([1, 4, 9, 16, 25]), 2))


<!-- dd:dd-q163 -->

### Problem 163 · independent

Write a function solve(z) that takes a 2-D PyTorch integer tensor and returns the subarray of rows that are STRICTLY INCREASING left to right (each entry greater than the one before it), preserving their original order. If no row qualifies, the result has zero rows.

**Expected output** — run the cell below once `solve` is right and it should print this.

```text
tensor([[1, 2, 3]])
```


In [ ]:
import torch as t

def solve(z):
    """Return only the strictly increasing rows of z."""
    return None


# Example run — the grader calls solve() with several arrays.
print(solve(t.tensor([[1, 2, 3], [3, 2, 1]])))


In [ ]:
# Did it work? Run this. (NameError → run the checker cell at
# the top of the notebook first: Runtime ▸ Run before.)
dd_check(163)


In [ ]:
#@title 💡 Solution — Problem 163
# Running this rebinds `solve` to the reference answer. Re-run your
# own cell before dd_check() again, or you are checking this one.
import torch as t

def solve(z):
    mask = t.all(t.diff(z, dim=1) > 0, dim=1)
    return z[mask]


print(solve(t.tensor([[1, 2, 3], [3, 2, 1]])))


#### Common mistakes

- **"cumsum needs a loop with a running variable."** — `t.cumsum` IS that
  loop, in compiled code. Same for the running max, min and product.
- **"diff returns the same length."** — One shorter per application: n
  numbers have n−1 gaps. `t.diff(x, n=k)` shrinks by k. Plan output shapes
  accordingly.
- **"Running maximum = amax with a dim."** — `amax(dim=...)` collapses the
  dimension to ONE value; the running version keeps the answer at every point.
  "So far" in the task text is the tell that you want a `cum*`.
- **"cummax returns a tensor."** — It returns a `(values, indices)` pair.
  Forgetting `.values` is the standard first mistake, and the error it causes
  shows up later, wherever the pair is finally used as a tensor.


<!-- dd:dd-kp-numpy-dot-matmul-patterns -->

## Dot products and matrix-multiply patterns

`numpy.dot-matmul-patterns`


<!-- dd:dd-seg-numpy-dot-matmul-patterns-0 -->

### the dot product — multiply, then sum


The **dot product** of two equal-length vectors — multiply corresponding
entries, add them up — is the atom that all of linear algebra's products are
built from. PyTorch spells it three interchangeable ways:

```python no-run
t.dot(a, b)      ==  a @ b  ==  (a * b).sum()
```

The third spelling is the important one conceptually: *dot = elementwise
multiply + reduction.* Holding the decomposition lets you build variants
(weighted dots, masked dots, batch dots) instead of hunting for a function
that may not exist.


In [ ]:
import torch as t

a = t.tensor([1.0, 2.0, 3.0])
b = t.tensor([4.0, -5.0, 6.0])

# The atom, three spellings — same number.
d = float(t.dot(a, b))
assert d == float(a @ b) == float((a * b).sum()) == 12.0
print("a * b (no sum yet):", a * b)
print("t.dot:", d, "| a @ b:", float(a @ b), "| (a*b).sum():",
      float((a * b).sum()))




Why: verifying the three dot spellings agree once buys permanent fluency:
when you see `(x * w).sum()` in someone's code, you now read "dot".


<!-- dd:dd-q37 -->

### Problem 37 · faded — your turn

Dot product of two vectors, as a plain float.

**Expected output** — run the cell below once `solve` is right and it should print this.

```text
12.0
```


In [ ]:
import torch as t

def solve(a, b):
    """The dot product of vectors a and b."""
    return float(t._____(a, b))


# Example run — the grader calls solve() with several different vector pairs,
# including edge cases. Your function must work for all of them.
a_example = t.tensor([1.0, 2.0, 3.0])
b_example = t.tensor([4.0, -5.0, 6.0])
print(solve(a_example, b_example))


In [ ]:
# Did it work? Run this. (NameError → run the checker cell at
# the top of the notebook first: Runtime ▸ Run before.)
dd_check(37)


In [ ]:
#@title 💡 Solution — Problem 37
# Running this rebinds `solve` to the reference answer. Re-run your
# own cell before dd_check() again, or you are checking this one.
import torch as t

def solve(a, b):
    return float(t.dot(a, b))


a_example = t.tensor([1.0, 2.0, 3.0])
b_example = t.tensor([4.0, -5.0, 6.0])
print(solve(a_example, b_example))


<!-- dd:dd-seg-numpy-dot-matmul-patterns-1 -->

### matrix @ vector — one dot per row


**Matrix @ vector** (`z @ v`, shapes (n, m) @ (m,)): one dot per row of z —
"each row dotted with v" in a single call. Result shape (n,). (Matrix @
matrix is one dot per (row, column) pair — the previous KP.)

Check one output by hand and the shape rule follows: matmul = a dot per
row, so an (n, m) matrix against a length-m vector yields n dots.


In [ ]:
import torch as t

# Matrix @ vector: row i of the result = (row i of z) . v
z = t.tensor([[1.0, 2.0],
              [3.0, 4.0]])
v = t.tensor([10.0, 1.0])
zv = z @ v
assert zv.tolist() == [12.0, 34.0]        # 1*10+2*1, 3*10+4*1
print(tuple(z.shape), "@", tuple(v.shape), "->", tuple(zv.shape), ":", zv)




Why: checking one output by hand (1·10 + 2·1 = 12) anchors "matmul = a dot
per row" — and predicts the result's shape (n,) without memorizing another
rule.


<!-- dd:dd-q144 -->

### Problem 144 · faded — your turn

Each row of z dotted with v.

**Expected output** — run the cell below once `solve` is right and it should print this.

```text
tensor([12., 34.])
```


In [ ]:
import torch as t

def solve(z, v):
    """Length-n array: entry i = (row i of z) . v."""
    return z _____ v


# Example run — the grader calls solve() with several inputs.
print(solve(t.tensor([[1.0, 2.0], [3.0, 4.0]]), t.tensor([10.0, 1.0])))


In [ ]:
# Did it work? Run this. (NameError → run the checker cell at
# the top of the notebook first: Runtime ▸ Run before.)
dd_check(144)


In [ ]:
#@title 💡 Solution — Problem 144
# Running this rebinds `solve` to the reference answer. Re-run your
# own cell before dd_check() again, or you are checking this one.
import torch as t

def solve(z, v):
    return z @ v


print(solve(t.tensor([[1.0, 2.0], [3.0, 4.0]]), t.tensor([10.0, 1.0])))


<!-- dd:dd-seg-numpy-dot-matmul-patterns-2 -->

### when @ doesn't fit — multiply, then reduce an axis


When the pattern you need is *not* one of the packaged shapes, the
decomposition rescues you. "Dot each row of `a` with the CORRESPONDING row
of `b`" (same shapes) is not `a @ b` — matmul dots every row with every
COLUMN. But per the atom: multiply elementwise, then reduce each row:

```python no-run
(a * b).sum(dim=1)          # row-wise dots
```

That multiply-then-reduce-an-axis maneuver covers the "batch of dots" tasks
— and it is the exact pattern einsum notation will name concisely in the
next course topic.


In [ ]:
import torch as t

# Row-wise dots of two SAME-SHAPE matrices: NOT a matmul.
p = t.tensor([[1.0, 2.0],
              [3.0, 4.0]])
q = t.tensor([[5.0, 6.0],
              [7.0, 8.0]])
row_dots = (p * q).sum(dim=1)
assert row_dots.tolist() == [17.0, 53.0]  # 1*5+2*6, 3*7+4*8
print("row-wise dots", row_dots)
print("p @ q would be something else entirely:")
print(p @ q)




Why: this case is deliberately a trap — `p @ q` runs on these square
matrices and returns the WRONG thing (full matrix product). Decomposing to
multiply+sum(axis=1) is the general escape whenever the packaged products
don't match the pairing you need.


<!-- dd:dd-q121 -->

### Problem 121 · faded — your turn

Dot each row of a with the corresponding row of b.

**Expected output** — run the cell below once `solve` is right and it should print this.

```text
tensor([11.])
```


In [ ]:
import torch as t

def solve(a, b):
    """Length-n array: entry i = (row i of a) . (row i of b)."""
    return (a * b).sum(dim=_____)


# Example run — the grader calls solve() with several pairs.
print(solve(t.tensor([[1.0, 2.0]]), t.tensor([[3.0, 4.0]])))


In [ ]:
# Did it work? Run this. (NameError → run the checker cell at
# the top of the notebook first: Runtime ▸ Run before.)
dd_check(121)


In [ ]:
#@title 💡 Solution — Problem 121
# Running this rebinds `solve` to the reference answer. Re-run your
# own cell before dd_check() again, or you are checking this one.
import torch as t

def solve(a, b):
    return t.einsum('ij,ij->i', a, b)


print(solve(t.tensor([[1.0, 2.0]]), t.tensor([[3.0, 4.0]])))


<!-- dd:dd-seg-numpy-dot-matmul-patterns-3 -->

### norms — a dot with itself, rooted


**Norms** are the same atom again: a vector's Euclidean length is
`t.sqrt(v @ v)`. Applied to a whole matrix's entries — √(sum of all
squares) — it's the **Frobenius norm**, `t.linalg.norm(z)` with no
arguments (as if the matrix were one long vector). Operator norms exist
behind `ord=`, but Frobenius is the drills' default meaning of "the norm".


In [ ]:
import torch as t

# Frobenius norm: sqrt of the sum of ALL squared entries.
f = t.tensor([[3.0, 4.0],
              [0.0, 0.0]])
assert t.linalg.norm(f) == 5.0
assert t.isclose(t.linalg.norm(f), t.sqrt((f * f).sum()))
print("norm", t.linalg.norm(f).item(), "| sqrt((f*f).sum())",
      t.sqrt((f * f).sum()).item())




Why: the first-principles spelling `t.sqrt((z * z).sum())` is
multiply+reduce again — the whole KP is one atom wearing different hats.


<!-- dd:dd-q5 -->

### Problem 5 · faded — your turn

The Frobenius norm of a matrix.

**Expected output** — run the cell below once `solve` is right and it should print this.

```text
tensor(5.)
```


In [ ]:
import torch as t

def solve(z):
    """Square root of the sum of squares of ALL entries of z."""
    return t.linalg._____(z)


# Example run — the grader calls solve() with several matrices.
example = t.tensor([[3.0, 4.0], [0.0, 0.0]])
print(solve(example))


In [ ]:
# Did it work? Run this. (NameError → run the checker cell at
# the top of the notebook first: Runtime ▸ Run before.)
dd_check(5)


In [ ]:
#@title 💡 Solution — Problem 5
# Running this rebinds `solve` to the reference answer. Re-run your
# own cell before dd_check() again, or you are checking this one.
import torch as t

def solve(z):
    return t.linalg.norm(z)


example = t.tensor([[3.0, 4.0], [0.0, 0.0]])
print(solve(example))


<!-- dd:dd-q514 -->

### Problem 514 · guided

Write a function solve(a, b) that takes two 1-D float tensors of the same length and returns their dot product as a plain Python float — but build it from the two steps it actually is: multiply the pairs, then sum the results. Seeing the pattern in the open is what lets you recognise it later in shapes that @ will not take.

**Expected output** — run the cell below once `solve` is right and it should print this.

```text
12.0
```


<details>
<summary>Hints</summary>

1. Do not reach for the built-in dot product — the question wants the two
   steps it is made of.
2. Multiply the pairs elementwise, then collapse the result to one number.
3. `float((a * b).sum())`.

</details>


In [ ]:
import torch as t

def solve(a, b):
    """Return the dot product built by hand: multiply, then sum."""
    return None


# Example run — the grader calls solve() with several different inputs,
# including edge cases. Your function must work for all of them.
example = (t.tensor([1.0, 2.0, 3.0]), t.tensor([4.0, -5.0, 6.0]))
print(solve(*example))


In [ ]:
# Did it work? Run this. (NameError → run the checker cell at
# the top of the notebook first: Runtime ▸ Run before.)
dd_check(514)


In [ ]:
#@title 💡 Solution — Problem 514
# Running this rebinds `solve` to the reference answer. Re-run your
# own cell before dd_check() again, or you are checking this one.
import torch as t

def solve(a, b):
    """Return the dot product built by hand: multiply, then sum."""
    return float((a * b).sum())


example = (t.tensor([1.0, 2.0, 3.0]), t.tensor([4.0, -5.0, 6.0]))
print(solve(*example))


<!-- dd:dd-q141 -->

### Problem 141 · independent

Write a function solve(a, b) that takes two square float matrices of the same shape (n, n) and returns a 1-D tensor of length n holding the DIAGONAL of the matrix product a @ b — without computing the full product (entry i is the dot product of row i of a with column i of b). Any approach producing those values passes, but the einsum form is the one worth learning.

**Expected output** — run the cell below once `solve` is right and it should print this.

```text
tensor([19., 50.])
```


In [ ]:
import torch as t

def solve(a, b):
    """Return diag(a @ b) as a 1-D array."""
    return None


# Example run — the grader calls solve() with several pairs.
example_a = t.tensor([[1.0, 2.0], [3.0, 4.0]])
example_b = t.tensor([[5.0, 6.0], [7.0, 8.0]])
print(solve(example_a, example_b))


In [ ]:
# Did it work? Run this. (NameError → run the checker cell at
# the top of the notebook first: Runtime ▸ Run before.)
dd_check(141)


In [ ]:
#@title 💡 Solution — Problem 141
# Running this rebinds `solve` to the reference answer. Re-run your
# own cell before dd_check() again, or you are checking this one.
import torch as t

def solve(a, b):
    return t.einsum('ij,ji->i', a, b)


example_a = t.tensor([[1.0, 2.0], [3.0, 4.0]])
example_b = t.tensor([[5.0, 6.0], [7.0, 8.0]])
print(solve(example_a, example_b))


<!-- dd:dd-q515 -->

### Problem 515 · independent

Write a function solve(x) that takes a 2-D float tensor with no all-zero rows and returns it with each row rescaled to length 1. A row's length is the square root of its dot product with itself — reduce along the columns, keep the axis so it broadcasts back, and divide.

**Expected output** — run the cell below once `solve` is right and it should print this.

```text
tensor([[0.6000, 0.8000],
        [0.0000, 1.0000]])
```


In [ ]:
import torch as t

def solve(x):
    """Scale every row to unit length."""
    return None


# Example run — the grader calls solve() with several different inputs,
# including edge cases. Your function must work for all of them.
example = t.tensor([[3.0, 4.0], [0.0, 2.0]])
print(solve(example))


In [ ]:
# Did it work? Run this. (NameError → run the checker cell at
# the top of the notebook first: Runtime ▸ Run before.)
dd_check(515)


In [ ]:
#@title 💡 Solution — Problem 515
# Running this rebinds `solve` to the reference answer. Re-run your
# own cell before dd_check() again, or you are checking this one.
import torch as t

def solve(x):
    """Scale every row to unit length."""
    return x / x.pow(2).sum(dim=1, keepdim=True).sqrt()


example = t.tensor([[3.0, 4.0], [0.0, 2.0]])
print(solve(example))


<!-- dd:dd-q95 -->

### Problem 95 · independent

Write a function solve(img) that takes an RGB image as a 3-D PyTorch tensor of shape (h, w, 3) and returns the 2-D grayscale version of shape (h, w), where each output pixel is the weighted sum 0.299*R + 0.587*G + 0.114*B of that pixel's three channels. Do not use Python loops.

**Expected output** — run the cell below once `solve` is right and it should print this.

```text
tensor([[255.,   0.],
        [  0.,   0.]])
```


In [ ]:
import torch as t

def solve(img):
    """Return the (h, w) grayscale of an (h, w, 3) RGB image."""
    return None


# Example run — the grader calls solve() with several images.
example = t.zeros((2, 2, 3))
example[0, 0] = t.tensor([255, 255, 255])
print(solve(example))


In [ ]:
# Did it work? Run this. (NameError → run the checker cell at
# the top of the notebook first: Runtime ▸ Run before.)
dd_check(95)


In [ ]:
#@title 💡 Solution — Problem 95
# Running this rebinds `solve` to the reference answer. Re-run your
# own cell before dd_check() again, or you are checking this one.
import torch as t

def solve(img):
    return img @ t.tensor([0.299, 0.587, 0.114])


example = t.zeros((2, 2, 3))
example[0, 0] = t.tensor([255, 255, 255])
print(solve(example))


#### Common mistakes

- **"Row-wise dots of two matrices = a @ b."** — Matmul dots every row with
  every COLUMN. Corresponding-rows pairing is elementwise-multiply + row
  reduction: `(a * b).sum(axis=1)`.
- **"The dot product is its own primitive."** — It's multiply + sum. Holding
  the decomposition lets you build variants (weighted dots, masked dots,
  batch dots) instead of hunting for a function that may not exist.
- **"norm of a matrix = largest row norm."** — Default `t.linalg.norm(z)` on
  2-D is FROBENIUS: all entries squared, summed, rooted — as if the matrix
  were one long vector. Operator norms exist behind `ord=`, but Frobenius is
  the drills' default meaning of "the norm".


<!-- dd:dd-kp-numpy-stack-concat-interleave -->

## Stacking, concatenating, interleaving

`numpy.stack-concat-interleave`


Combining arrays into one splits on a single question: **does the result have
a NEW axis, or grow an EXISTING one?**

- **Grow an existing dimension — `t.cat` and its 2-D shorthands.**
  `t.vstack([a, b])` stacks rows (b's rows below a's);
  `t.hstack([a, b])` extends rows sideways. Shapes must agree on the other
  axis; the combined axis just adds up. General form:
  `t.cat([a, b], dim=k)` — the shorthands are that call with `k` fixed.


In [ ]:
import torch as t

pair = [t.tensor([[1, 2]]), t.tensor([[3, 4]])]
print("dim=0 (taller):", t.cat(pair, dim=0).shape, t.cat(pair, dim=0).tolist())
print("dim=1 (wider): ", t.cat(pair, dim=1).shape, t.cat(pair, dim=1).tolist())
print("vstack is dim=0:", t.equal(t.vstack(pair), t.cat(pair, dim=0)))



- **Create a new axis — `t.stack`.**
  `t.stack([a, b], axis=0)` piles k same-shape arrays into a (k, …) array.
  Nothing merges; you gain a dimension. This is the bridge to *reductions
  over the pile*: the elementwise average of two arrays is
  `t.stack([a, b]).mean(axis=0)` — stack, then reduce the new axis. Any
  "combine k arrays by taking the elementwise mean/max/median" is this
  two-step.
- **Interleave — stack + reshape, or strided assignment.**
  Alternating elements `[a0, b0, a1, b1, …]` has two idiomatic spellings:
  - `t.column_stack((a, b)).ravel()` — pair up (each row `[a_i, b_i]`),
    then read row-major: the pairs unroll in exactly alternating order.
    (Reshape's fill order doing real work!)
  - Preallocate and stride: `out[0::2] = a; out[1::2] = b` — allocate the
    full-length result, then write each source into its residue class.
    Generalizes cleanly to 3+ sources (`0::3`, `1::3`, `2::3`) and to
    "insert nz zeros between entries" (`out[::nz+1] = z` into a zeros
    canvas).

Choosing: piles that keep identity → stack; seams along an axis → concat
family; alternating patterns → column_stack+ravel or strided slots.


Task: stack two matrices vertically and horizontally; average them
elementwise via a stack; interleave two vectors.


In [ ]:
import torch as t

a = t.tensor([[1.0, 2.0],
              [3.0, 4.0]])
b = t.tensor([[5.0, 6.0],
              [7.0, 8.0]])

# Grow axis 0 (rows below) / axis 1 (columns to the right).
v = t.vstack([a, b])
h = t.hstack([a, b])
assert v.shape == (4, 2) and h.shape == (2, 4)

# NEW axis then reduce it: elementwise average of the two arrays.
piled = t.stack([a, b], dim=0)        # shape (2, 2, 2) — nothing merged
assert piled.shape == (2, 2, 2)
avg = piled.mean(dim=0)                # collapse the pile
assert avg.tolist() == [[3.0, 4.0], [5.0, 6.0]]

# Interleave two vectors: pair rows, then row-major ravel unrolls
# them alternately.
x = t.tensor([1, 3, 5])
y = t.tensor([2, 4, 6])
inter = t.column_stack((x, y)).ravel()
assert inter.tolist() == [1, 2, 3, 4, 5, 6]

# Same result by strided assignment — the form that scales to 3+ streams.
out = t.empty(6, dtype=x.dtype)
out[0::2] = x
out[1::2] = y
assert out.tolist() == [1, 2, 3, 4, 5, 6]
print("vstack", tuple(v.shape), "| hstack", tuple(h.shape),
      "| stack", tuple(piled.shape), "<- stack ADDS an axis")
print("averaged over the new axis:")
print(avg)
print("interleaved:", inter, "| by strided assignment:", out)




Why each step:

1. Track shapes: vstack (2,2)+(2,2)→(4,2) grew an axis; stack →(2,2,2) added
   one. The shape arithmetic is the reliable way to tell which operation a
   task describes.
2. stack-then-reduce turns "elementwise average/max of k arrays" into the
   axis machinery you already own — no dedicated function needed, and it
   generalizes from mean to any reduction.
3. Both interleave spellings matter: column_stack+ravel is elegant for two
   streams; residue-class assignment (`empty` first — safe here because
   every slot gets written) reads mechanically but handles any number of
   streams and irregular spacings.


<!-- dd:dd-q84 -->

### Problem 84 · faded — your turn

Elementwise average of two same-shape arrays, via a new axis.

**Expected output** — run the cell below once `solve` is right and it should print this.

```text
tensor([[2., 4.]])
```


In [ ]:
import torch as t

def solve(a, b):
    """Elementwise average: stack on a new axis, then reduce it."""
    return t._____([a, b], _____=0)._____(_____=0)


# Example run — the grader calls solve() with several pairs.
example_a = t.tensor([[1.0, 2.0]])
example_b = t.tensor([[3.0, 6.0]])
print(solve(example_a, example_b))


In [ ]:
# Did it work? Run this. (NameError → run the checker cell at
# the top of the notebook first: Runtime ▸ Run before.)
dd_check(84)


In [ ]:
#@title 💡 Solution — Problem 84
# Running this rebinds `solve` to the reference answer. Re-run your
# own cell before dd_check() again, or you are checking this one.
import torch as t

def solve(a, b):
    return t.stack([a, b], dim=0).mean(dim=0)


example_a = t.tensor([[1.0, 2.0]])
example_b = t.tensor([[3.0, 6.0]])
print(solve(example_a, example_b))


<!-- dd:dd-q146 -->

### Problem 146 · guided

Write a function solve(a, b, c) that takes three 1-D PyTorch tensors of the same length n and returns a single tensor of length 3n interleaving them position by position: the result reads a[0], b[0], c[0], a[1], b[1], c[1], and so on. Do not use Python loops.

**Expected output** — run the cell below once `solve` is right and it should print this.

```text
tensor([  1,  10, 100,   2,  20, 200])
```


<details>
<summary>Hints</summary>

1. Three streams interleaved position by position — the pair-and-ravel trick
   still works, but the strided form is clearer: what are the three residue
   classes?
2. Allocate the result (`t.empty(3 * n, dtype=...)` — dtype from the inputs
   via `t.result_type`), then one slice assignment per stream.
3. `out[0::3] = a; out[1::3] = b; out[2::3] = c`.

</details>


In [ ]:
import torch as t

def solve(a, b, c):
    """Return a, b, c interleaved elementwise into one length-3n array."""
    return None


# Example run — the grader calls solve() with several triples.
print(solve(t.tensor([1, 2]), t.tensor([10, 20]), t.tensor([100, 200])))


In [ ]:
# Did it work? Run this. (NameError → run the checker cell at
# the top of the notebook first: Runtime ▸ Run before.)
dd_check(146)


In [ ]:
#@title 💡 Solution — Problem 146
# Running this rebinds `solve` to the reference answer. Re-run your
# own cell before dd_check() again, or you are checking this one.
import torch as t
def solve(a, b, c):
    dtype = t.promote_types(t.result_type(a, b), c.dtype)
    out = t.empty(a.numel() * 3, dtype=dtype)
    out[0::3] = a
    out[1::3] = b
    out[2::3] = c
    return out


print(solve(t.tensor([1, 2]), t.tensor([10, 20]), t.tensor([100, 200])))


<!-- dd:dd-q89 -->

### Problem 89 · independent

Write a function solve(a, b) that takes two 1-D PyTorch tensors of the same length n and returns a single 1-D tensor of length 2n whose entries alternate between a and b starting with a[0]: [a[0], b[0], a[1], b[1], ...]. Do not use Python loops.

**Expected output** — run the cell below once `solve` is right and it should print this.

```text
tensor([1, 2, 3, 4, 5, 6])
```


In [ ]:
import torch as t

def solve(a, b):
    """Return a and b interleaved: a[0], b[0], a[1], b[1], ..."""
    return None


# Example run — the grader calls solve() with several pairs.
print(solve(t.tensor([1, 3, 5]), t.tensor([2, 4, 6])))


In [ ]:
# Did it work? Run this. (NameError → run the checker cell at
# the top of the notebook first: Runtime ▸ Run before.)
dd_check(89)


In [ ]:
#@title 💡 Solution — Problem 89
# Running this rebinds `solve` to the reference answer. Re-run your
# own cell before dd_check() again, or you are checking this one.
import torch as t

def solve(a, b):
    return t.ravel(t.column_stack((a, b)))


print(solve(t.tensor([1, 3, 5]), t.tensor([2, 4, 6])))


<!-- dd:dd-q238 -->

### Problem 238 · independent

Write a function solve(a, b) that takes two 2-D PyTorch tensors a and b with the same shape. It should return a tuple of two tensors: the first is the vertical combination, with the rows of b stacked below the rows of a; the second is the horizontal combination, with b placed side by side to the right of a. Do not modify the input tensors.

**Expected output** — run the cell below once `solve` is right and it should print this.

```text
(tensor([[1., 2.],
        [3., 4.],
        [5., 6.],
        [7., 8.]]), tensor([[1., 2., 5., 6.],
        [3., 4., 7., 8.]]))
```


In [ ]:
import torch as t

def solve(a, b):
    """Return (vertical combination of a and b, horizontal combination of a and b)."""
    return None


# Example run — the grader calls solve() with several different array pairs,
# including edge cases. Your function must work for all of them.
a = t.tensor([[1.0, 2.0], [3.0, 4.0]])
b = t.tensor([[5.0, 6.0], [7.0, 8.0]])
print(solve(a, b))


In [ ]:
# Did it work? Run this. (NameError → run the checker cell at
# the top of the notebook first: Runtime ▸ Run before.)
dd_check(238)


In [ ]:
#@title 💡 Solution — Problem 238
# Running this rebinds `solve` to the reference answer. Re-run your
# own cell before dd_check() again, or you are checking this one.
import torch as t

def solve(a, b):
    return t.vstack([a, b]), t.hstack([a, b])


a = t.tensor([[1.0, 2.0], [3.0, 4.0]])
b = t.tensor([[5.0, 6.0], [7.0, 8.0]])
print(solve(a, b))


<!-- dd:dd-q159 -->

### Problem 159 · independent

Write a function solve(z, nz) that takes a 1-D PyTorch tensor z and a non-negative integer nz, and returns a new tensor in which nz zeros are interleaved between every pair of consecutive entries of z: the result has length len(z) + (len(z) - 1) * nz, with z's values at positions 0, nz+1, 2*(nz+1), ...

**Expected output** — run the cell below once `solve` is right and it should print this.

```text
tensor([1., 0., 0., 0., 2., 0., 0., 0., 3., 0., 0., 0., 4., 0., 0., 0., 5.])
```


In [ ]:
import torch as t

def solve(z, nz):
    """Return z with nz zeros inserted between consecutive entries."""
    return None


# Example run — the grader calls solve() with several inputs.
print(solve(t.tensor([1, 2, 3, 4, 5]), 3))


In [ ]:
# Did it work? Run this. (NameError → run the checker cell at
# the top of the notebook first: Runtime ▸ Run before.)
dd_check(159)


In [ ]:
#@title 💡 Solution — Problem 159
# Running this rebinds `solve` to the reference answer. Re-run your
# own cell before dd_check() again, or you are checking this one.
import torch as t

def solve(z, nz):
    out = t.zeros(len(z) + (len(z) - 1) * nz)
    out[:: nz + 1] = z
    return out


print(solve(t.tensor([1, 2, 3, 4, 5]), 3))


#### Common mistakes

- **"stack and concatenate are synonyms."** — concatenate grows an existing
  axis (no new dimension); stack creates a new one. (2,3)+(2,3): concat
  axis-0 → (4,3); stack → (2,2,3). The task's result shape tells you which.
- **"Interleaving needs a Python loop."** — Either pair-and-ravel
  (column_stack + row-major flatten) or strided slice assignment. Both are
  single-pass, loop-free.
- **"t.empty is dangerous here."** — It's uninitialized memory, which is
  fine EXACTLY when every slot gets written before any read — as in the
  residue-class pattern. If any slot might stay untouched (the zeros-between
  drill!), start from `t.zeros` instead.


<!-- dd:dd-kp-numpy-onehot-bincount -->

## Labels — one-hot encoding and bincount

`numpy.onehot-bincount`


Integer **class labels** (0, 1, …, K−1) have two canonical transformations,
and both are one-liners once you see the trick.

**One-hot encoding: `t.eye(k)[labels]`.**
A one-hot row for class c is a length-k vector of zeros with a 1 in slot c —
which is precisely **row c of the k×k identity matrix**. So encoding a whole
label vector is a lookup-table read (fancy-indexing KP) where the table is
`t.eye(k)`: each label picks its identity row, and the result stacks them
into shape (len(labels), k). Need integers instead of floats? Build the table
that way: `t.eye(k, dtype=int)`. Don't know k? The labels tell you:
`k = labels.max() + 1`.

**Counting labels: `t.bincount(labels)`.**
Returns an array where entry v is *how many times value v occurs* — a
histogram over the non-negative integers 0..max. Unlike `t.unique`'s counts
(which list only values that appear), bincount's output is **dense**: absent
values get an explicit 0, and the position IS the value. That density powers
compositions:

- **Mode** (most frequent value): `t.bincount(x).argmax()` — and because
  argmax breaks ties at the first index, ties resolve to the SMALLEST value
  automatically.
- **Weighted sums per class**: `t.bincount(labels, weights=v)` sums v's
  entries per class — grouped aggregation in one call (the applied lesson
  builds on this).

The two are inverses in spirit: one-hot *spreads* a label into a row;
bincount *collapses* a label vector into per-class totals. (Indeed
`onehot.sum(axis=0)` equals the bincount.)


Task: one-hot a label vector; count label occurrences; find the mode with
smallest-value tie-breaking.


In [ ]:
import torch as t

labels = t.tensor([0, 2, 1, 2])
k = 3

# One-hot: the identity matrix as lookup table, labels as row selectors.
onehot = t.eye(k)[labels]
assert onehot.shape == (4, 3)
assert onehot.tolist() == [[1.0, 0.0, 0.0],
                           [0.0, 0.0, 1.0],
                           [0.0, 1.0, 0.0],
                           [0.0, 0.0, 1.0]]

# bincount: entry v = multiplicity of v. Dense — class 0,1,2 all present.
counts = t.bincount(labels)
assert counts.tolist() == [1, 1, 2]

# The two views agree: summing one-hot rows counts the classes.
assert t.equal(onehot.sum(dim=0), counts)

# Mode with smallest-on-tie: argmax over the dense counts.
x = t.tensor([3, 1, 3, 2, 3, 1])
mode = int(t.bincount(x).argmax())
assert mode == 3
# Tie case: 1 and 2 both appear twice -> argmax hits index 1 first.
assert int(t.bincount(t.tensor([1, 2, 1, 2])).argmax()) == 1
print("labels", labels)
print(onehot)
print("bincount", counts, "| one-hot columns summed", onehot.sum(dim=0))
print("mode of", x.tolist(), "is", mode)




Why each step:

1. Seeing `t.eye(k)[labels]` as "lookup table = identity" connects three
   prior KPs (constructors, fancy indexing) into an idiom you can re-derive
   under exam conditions rather than memorize.
2. The `onehot.sum(axis=0) == bincount` identity is a genuine consistency
   check — worth one assert when correctness matters.
3. The tie-break behavior isn't luck: bincount's index-is-value layout plus
   argmax's first-occurrence rule together GUARANTEE smallest-value ties.
   Reading composition behavior off the parts is the skill.


<!-- dd:dd-q93 -->

### Problem 93 · faded — your turn

One-hot rows for a label vector, class count given.

**Expected output** — run the cell below once `solve` is right and it should print this.

```text
tensor([[1., 0., 0.],
        [0., 0., 1.],
        [0., 1., 0.],
        [0., 0., 1.]])
```


In [ ]:
import torch as t

def solve(labels, k):
    """(len(labels), k) one-hot matrix: row i encodes labels[i]."""
    return t._____(k)[labels]


# Example run — the grader calls solve() with several inputs.
print(solve(t.tensor([0, 2, 1, 2]), 3))


In [ ]:
# Did it work? Run this. (NameError → run the checker cell at
# the top of the notebook first: Runtime ▸ Run before.)
dd_check(93)


In [ ]:
#@title 💡 Solution — Problem 93
# Running this rebinds `solve` to the reference answer. Re-run your
# own cell before dd_check() again, or you are checking this one.
import torch as t

def solve(labels, k):
    return t.eye(k)[labels]


print(solve(t.tensor([0, 2, 1, 2]), 3))


<!-- dd:dd-q134 -->

### Problem 134 · guided

Write a function solve(x) that takes a 1-D PyTorch tensor x of non-negative integers and returns the value that occurs most frequently in x. If several values are tied for the highest count, return the smallest of those values. Return the result as a plain Python int. Do not use Python loops.

**Expected output** — run the cell below once `solve` is right and it should print this.

```text
3
```


<details>
<summary>Hints</summary>

1. The most frequent value of a non-negative integer array, smallest on ties
   — which counting tool gives you position-is-value output?
2. Once counts are dense, "most frequent value" is the INDEX of the largest
   count.
3. `int(t.bincount(x).argmax())` — convince yourself why ties come out
   smallest for free.

</details>


In [ ]:
import torch as t

def solve(x):
    """Return the most frequent value in x, breaking ties by choosing the smallest value."""
    return None


# Example run — the grader calls solve() with several different arrays,
# including edge cases. Your function must work for all of them.
example = t.tensor([3, 1, 3, 2, 3, 1])
print(solve(example))


In [ ]:
# Did it work? Run this. (NameError → run the checker cell at
# the top of the notebook first: Runtime ▸ Run before.)
dd_check(134)


In [ ]:
#@title 💡 Solution — Problem 134
# Running this rebinds `solve` to the reference answer. Re-run your
# own cell before dd_check() again, or you are checking this one.
import torch as t

def solve(x):
    return int(t.bincount(x).argmax())


example = t.tensor([3, 1, 3, 2, 3, 1])
print(solve(example))


<!-- dd:dd-q150 -->

### Problem 150 · independent

Write a function solve(y) that takes a 1-D integer label vector whose values lie in {0, ..., C-1} where C = y.max() + 1 (every class up to the max may appear), and returns a 2-D INTEGER tensor of shape (len(y), C) whose row i is the one-hot encoding of y[i]. Unlike a float one-hot, the dtype here must be an integer kind, and C is inferred from y itself.

**Expected output** — run the cell below once `solve` is right and it should print this.

```text
tensor([[1, 0, 0],
        [0, 0, 1],
        [0, 1, 0],
        [0, 0, 1],
        [1, 0, 0]])
```


In [ ]:
import torch as t

def solve(y):
    """Return the integer one-hot matrix for y, width y.max()+1."""
    return None


# Example run — the grader calls solve() with several label vectors.
print(solve(t.tensor([0, 2, 1, 2, 0])))


In [ ]:
# Did it work? Run this. (NameError → run the checker cell at
# the top of the notebook first: Runtime ▸ Run before.)
dd_check(150)


In [ ]:
#@title 💡 Solution — Problem 150
# Running this rebinds `solve` to the reference answer. Re-run your
# own cell before dd_check() again, or you are checking this one.
import torch as t

def solve(y):
    c = y.max() + 1
    return t.eye(c, dtype=t.int64)[y]


print(solve(t.tensor([0, 2, 1, 2, 0])))


<!-- dd:dd-q124 -->

### Problem 124 · independent

Write a function solve(z) that takes a 2-D float tensor with a unique maximum in each row and returns a same-shape float tensor containing 1.0 at each row's maximum position and 0.0 everywhere else (a per-row argmax one-hot). Do not use Python loops.

**Expected output** — run the cell below once `solve` is right and it should print this.

```text
tensor([[0., 1.],
        [1., 0.]])
```


In [ ]:
import torch as t

def solve(z):
    """Return a one-hot array marking each row's maximum with 1.0."""
    return None


# Example run — the grader calls solve() with several arrays.
example = t.tensor([[0.1, 0.9], [0.8, 0.2]])
print(solve(example))


In [ ]:
# Did it work? Run this. (NameError → run the checker cell at
# the top of the notebook first: Runtime ▸ Run before.)
dd_check(124)


In [ ]:
#@title 💡 Solution — Problem 124
# Running this rebinds `solve` to the reference answer. Re-run your
# own cell before dd_check() again, or you are checking this one.
import torch as t

def solve(z):
    out = t.zeros_like(z)
    out[t.arange(z.shape[0]), z.argmax(dim=1)] = 1.0
    return out


example = t.tensor([[0.1, 0.9], [0.8, 0.2]])
print(solve(example))


<!-- dd:dd-q172 -->

### Problem 172 · independent

Write a function solve(z) that takes a 2-D tensor of small NON-NEGATIVE integers and returns a 1-D tensor with each row's MODE — its most frequently occurring value — breaking ties by choosing the SMALLEST tied value.

**Expected output** — run the cell below once `solve` is right and it should print this.

```text
tensor([2, 3])
```


In [ ]:
import torch as t

def solve(z):
    """Return each row's most frequent value (ties -> smallest value)."""
    return None


# Example run — the grader calls solve() with several arrays.
print(solve(t.tensor([[1, 2, 2], [3, 3, 0]])))


In [ ]:
# Did it work? Run this. (NameError → run the checker cell at
# the top of the notebook first: Runtime ▸ Run before.)
dd_check(172)


In [ ]:
#@title 💡 Solution — Problem 172
# Running this rebinds `solve` to the reference answer. Re-run your
# own cell before dd_check() again, or you are checking this one.
import torch as t
def solve(z):
    m = int(z.max()) + 1
    return t.stack([t.bincount(row, minlength=m).argmax() for row in z])


print(solve(t.tensor([[1, 2, 2], [3, 3, 0]])))


#### Common mistakes

- **"One-hot needs a loop setting out[i, labels[i]] = 1."** — That loop is
  exactly what `t.eye(k)[labels]` performs in one vectorized gather. (The
  explicit-canvas form does have its place — see the per-row variant in
  q124.)
- **"bincount == unique counts."** — unique's counts are COMPACT (only values
  present, paired with a values array); bincount is DENSE (every integer
  0..max gets a slot, position = value). Mode-finding and class-vector tasks
  want the dense layout.
- **"bincount works on any integers."** — Non-negative only; negatives raise.
  Shift first (`x - x.min()`) if the data can dip below zero, and shift the
  interpretation back afterwards.


<!-- dd:dd-kp-numpy-topk-selection -->

## Top-k selection — topk vs sort

`numpy.topk-selection`


"The k largest values" does not require sorting everything, and PyTorch packs
the whole operation into one call:

> **`t.topk(z, k)`** → a `(values, indices)` pair holding the k largest
> entries, **largest first**.

Both halves come back together, so the two questions a top-k task can ask —
*which values* and *at which positions* — are answered by reading `.values`
or `.indices` off the same result. Cost is O(len + k log k) rather than a
full O(len log len) sort; for small k on a big tensor that gap is what the
word "efficiently" in a task is pointing at.

Two knobs matter:

- **`largest=False`** flips it to the k SMALLEST.
- **`sorted=False`** drops the ordering guarantee, returning the top-k SET
  slightly cheaper — use it when you only need membership.

Because the default is already sorted descending, "largest first" needs no
follow-up step, and ascending order is just a `.flip(0)`:

```python no-run
t.topk(z, k).values.flip(0)      # k largest, ascending
t.topk(z, k, dim=1).indices      # per-row top-k indices, largest first
```

Per-row top-k on matrices: `topk` takes `dim=`, so `t.topk(z, k, dim=1)`
gives a (rows, k) block directly. For the k-th largest *single* value,
`t.kthvalue(z, k)` is the one-element cousin.

Decision rule: **full order over everything → sort/argsort; only the top k →
topk** (and `.flip(0)` if the task wants them ascending).


Task: the 2 largest values in ascending order; then their indices ordered
largest-first.


In [ ]:
import torch as t

z = t.tensor([5, 1, 9, 3, 7])

# topk returns both halves at once, largest first.
top = t.topk(z, 2)
assert top.values.tolist() == [9, 7]
assert top.indices.tolist() == [2, 4]

# The task wants ascending — reverse the (already sorted) k values.
assert top.values.flip(0).tolist() == [7, 9]

# Index version: which POSITIONS hold the top-2, largest first?
w = t.tensor([5.0, 9.0, 1.0, 7.0])
ordered = t.topk(w, 2).indices
assert ordered.tolist() == [1, 3]
assert w[ordered].tolist() == [9.0, 7.0]
print("z", z, "-> topk", top)
print("ascending instead:", top.values.flip(0))
print("w", w, "-> top-2 positions", ordered, "-> values", w[ordered])




Why each step:

1. Reading `.values` and `.indices` off one result is the habit to build —
   NumPy needed two separate calls (`partition` and `argpartition`) that
   could disagree; here they cannot.
2. `flip(0)` costs nothing on k elements. Reaching for a second `sort` is the
   common reflex and is pure waste, since topk already ordered them.
3. Only k elements are ever ordered; that asymmetry (linear scan + tiny sort)
   is the entire efficiency argument, and stating it is usually what a
   drill's "efficiently" phrasing wants.


<!-- dd:dd-q206 -->

### Problem 206 · faded — your turn

The n largest values, ascending, efficiently.

**Expected output** — run the cell below once `solve` is right and it should print this.

```text
tensor([7, 9])
```


In [ ]:
import torch as t

def solve(z, n):
    """n largest values, ascending — topk gives them descending."""
    return t.topk(z, n).values._____(0)


# Example run — the grader calls solve() with several inputs.
print(solve(t.tensor([5, 1, 9, 3, 7]), 2))


In [ ]:
# Did it work? Run this. (NameError → run the checker cell at
# the top of the notebook first: Runtime ▸ Run before.)
dd_check(206)


In [ ]:
#@title 💡 Solution — Problem 206
# Running this rebinds `solve` to the reference answer. Re-run your
# own cell before dd_check() again, or you are checking this one.
import torch as t
def solve(z, n):
    return t.topk(z, n).values.flip(0)


print(solve(t.tensor([5, 1, 9, 3, 7]), 2))


<!-- dd:dd-q526 -->

### Problem 526 · guided

Write a function solve(z, k) that takes a 1-D tensor z with distinct values and returns a tuple (values, indices) holding the k SMALLEST entries of z, smallest first, together with the positions they came from. One call to t.topk answers both halves: a keyword argument flips it to the small end, and the ordering it returns by default is already the one asked for here. Do not sort the whole tensor.

**Expected output** — run the cell below once `solve` is right and it should print this.

```text
(tensor([1., 3.]), tensor([1, 3]))
```


<details>
<summary>Hints</summary>

1. Same call, other end of the range. `topk` takes a keyword that decides
   which end it keeps — no negation, no sort.
2. `largest=False`. And with `sorted` left at its default the k smallest come
   back ascending already, so there is nothing to flip here.
3. `small = t.topk(z, k, largest=False)`, then return
   `(small.values, small.indices)` — both halves off the one result.

</details>


In [ ]:
import torch
import torch as t

def solve(z, k):
    """(k smallest values, their positions) — smallest first."""
    return None


# Example run — the grader calls solve() with several different inputs,
# including edge cases. Your function must work for all of them.
example = (t.tensor([5.0, 1.0, 9.0, 3.0]), 2)
print(solve(*example))


In [ ]:
# Did it work? Run this. (NameError → run the checker cell at
# the top of the notebook first: Runtime ▸ Run before.)
dd_check(526)


In [ ]:
#@title 💡 Solution — Problem 526
# Running this rebinds `solve` to the reference answer. Re-run your
# own cell before dd_check() again, or you are checking this one.
import torch
import torch as t

def solve(z, k):
    """(k smallest values, their positions) — smallest first."""
    small = t.topk(z, k, largest=False)
    return (small.values, small.indices)


example = (t.tensor([5.0, 1.0, 9.0, 3.0]), 2)
print(solve(*example))


<!-- dd:dd-q187 -->

### Problem 187 · independent

Write a function solve(z, k) that takes a 2-D float tensor with distinct values in each row and returns an INTEGER mask of the same shape containing 1 at the positions of each row's k largest values and 0 elsewhere.

**Expected output** — run the cell below once `solve` is right and it should print this.

```text
tensor([[0, 1, 1]])
```


In [ ]:
import torch as t

def solve(z, k):
    """Return a 0/1 mask marking each row's k largest entries."""
    return None


# Example run — the grader calls solve() with several inputs.
print(solve(t.tensor([[0.1, 0.9, 0.5]]), 2))


In [ ]:
# Did it work? Run this. (NameError → run the checker cell at
# the top of the notebook first: Runtime ▸ Run before.)
dd_check(187)


In [ ]:
#@title 💡 Solution — Problem 187
# Running this rebinds `solve` to the reference answer. Re-run your
# own cell before dd_check() again, or you are checking this one.
import torch as t
def solve(z, k):
    idx = t.topk(z, k, dim=1).indices
    mask = t.zeros_like(z, dtype=t.int64)
    mask.scatter_(1, idx, 1)
    return mask


print(solve(t.tensor([[0.1, 0.9, 0.5]]), 2))


<!-- dd:dd-q194 -->

### Problem 194 · independent

Write a function solve(z, k) that takes a 2-D float tensor with distinct values within each row and returns an (rows, k) integer tensor where row i holds the column indices of that row's k largest values, ordered by DESCENDING value (largest first).

**Expected output** — run the cell below once `solve` is right and it should print this.

```text
tensor([[1, 0]])
```


In [ ]:
import torch as t

def solve(z, k):
    """Return per-row indices of the k largest values, largest first."""
    return None


# Example run — the grader calls solve() with several inputs.
print(solve(t.tensor([[0.4, 0.9, 0.1]]), 2))


In [ ]:
# Did it work? Run this. (NameError → run the checker cell at
# the top of the notebook first: Runtime ▸ Run before.)
dd_check(194)


In [ ]:
#@title 💡 Solution — Problem 194
# Running this rebinds `solve` to the reference answer. Re-run your
# own cell before dd_check() again, or you are checking this one.
import torch as t
def solve(z, k):
    return t.topk(z, k, dim=1).indices


print(solve(t.tensor([[0.4, 0.9, 0.1]]), 2))


#### Common mistakes

- **"Top-k requires sorting the tensor."** — `topk` finds and orders only k
  entries. On a million elements with k=10 that's the difference between one
  pass and a full N log N shuffle.
- **"topk returns them smallest-first."** — Largest first by default. Pass
  `largest=False` for the other end, and `.flip(0)` when you want the k
  largest in ascending order.
- **"You need argsort to get top-k indices."** — `.indices` comes back from
  the same call, already ordered by value. NumPy's separate
  partition/argpartition dance has no equivalent here, and reproducing it is
  strictly more work.


<!-- dd:dd-kp-numpy-inplace-out -->

## In-place operations and the trailing underscore

`numpy.inplace-out`


<!-- dd:dd-seg-numpy-inplace-out-0 -->

### the trailing underscore


Most PyTorch expressions allocate a fresh tensor per step. Usually fine — but
"in place" tasks (and memory-tight code) need the alternatives.

PyTorch marks in-place operations with a **trailing underscore**: `add_`,
`mul_`, `div_`, `neg_`, `clamp_`, `copy_`. Every one of them writes into the
existing buffer and returns that same tensor. Augmented operators (`x += 1`,
`x *= 2`) are the operator spelling of the same thing, where their written-out
forms (`x = x + 1`) allocate a new tensor and rebind the name.

The underscore is the whole signal, and it is worth trusting: a method without
one **never** modifies its receiver. `x.sort()` returns a sorted copy and
leaves `x` alone — there is no `sort_`, so sorting a tensor in place means
copying the sorted values back with `x.copy_(...)`.

The mirror-image rule from the slicing KP still applies: "do not modify the
input" → clone first. This KP is the deliberate OPPOSITE — recognize which
contract a task states before choosing tools.


In [ ]:
import torch as t

# Trailing underscore: same buffer, values doubled.
x = t.tensor([3.0, 1.0, 2.0])
x.mul_(2)
assert x.tolist() == [6.0, 2.0, 4.0]

# No underscore: sort() returns a (values, indices) pair and x is untouched.
result = x.sort()
assert x.tolist() == [6.0, 2.0, 4.0]
assert result.values.tolist() == [2.0, 4.0, 6.0]

# So an in-place sort is "sort, then copy the values back into the buffer".
print("after mul_(2):", x)
x.copy_(x.sort().values)
assert x.tolist() == [2.0, 4.0, 6.0]
print("after copy_(sorted values):", x)




Why: the underscore is a contract, not a style. `x.sort()` looks like it
should sort x — in NumPy the same spelling does — and here it quietly does
not.


<!-- dd:dd-q235 -->

### Problem 235 · faded — your turn

Ascending order, in place — the passed-in object itself must change.

**Expected output** — run the cell below once `solve` is right and it should print this.

```text
tensor([1., 2., 3.])
```


In [ ]:
import torch as t

def solve(x):
    """Sort x itself (no new tensor), then return it."""
    x._____(x._____().values)
    return x


# Example run — the grader calls solve() with several different arrays,
# including edge cases, and checks the contents of the array itself
# after the call. Your function must work for all of them.
example = t.tensor([3.0, 1.0, 2.0])
solve(example)
print(example)


In [ ]:
# Did it work? Run this. (NameError → run the checker cell at
# the top of the notebook first: Runtime ▸ Run before.)
dd_check(235)


In [ ]:
#@title 💡 Solution — Problem 235
# Running this rebinds `solve` to the reference answer. Re-run your
# own cell before dd_check() again, or you are checking this one.
import torch as t
def solve(x):
    x.copy_(x.sort().values)
    return x


example = t.tensor([3.0, 1.0, 2.0])
solve(example)
print(example)


<!-- dd:dd-seg-numpy-inplace-out-1 -->

### z[:] = expr — writing INTO the buffer


The subtle one: **whole-array in-place assignment.** `z = z[p]` REBINDS the
local name — the caller's array is untouched. `z[:] = z[p]` writes the
values INTO the existing buffer through a full-array slice, so every other
reference to that array sees the change. "Modify the array object passed
in" is this spelling. (Safe with `z[p]` on the right because fancy indexing
copies first.)


In [ ]:
import torch as t

# In-place row permutation: write INTO the buffer via z[:].
z = t.arange(6).reshape(3, 2)
alias = z                        # a second reference to the same buffer
p = t.tensor([2, 0, 1])
z[:] = z[p]                      # rebinding (z = z[p]) would NOT affect alias
assert alias.tolist() == [[4, 5], [0, 1], [2, 3]]
print("alias sees the permutation too:")
print(alias)




Why: the `alias` variable is the proof of what "in place" means — both
names watch the same memory, so only the `z[:] =` spelling changes what
`alias` sees. This distinction is the entire point of the drill family.


<!-- dd:dd-q138 -->

### Problem 138 · faded — your turn

Reorder rows by permutation p, in place.

**Expected output** — run the cell below once `solve` is right and it should print this.

```text
tensor([[4, 5],
        [0, 1],
        [2, 3]])
```


In [ ]:
import torch as t

def solve(z, p):
    """Row i becomes old row p[i] — INSIDE z's own buffer."""
    z_____ = z[p]
    return z


# Example run — the grader calls solve() with several inputs.
example = t.arange(6).reshape(3, 2)
print(solve(example, t.tensor([2, 0, 1])))


In [ ]:
# Did it work? Run this. (NameError → run the checker cell at
# the top of the notebook first: Runtime ▸ Run before.)
dd_check(138)


In [ ]:
#@title 💡 Solution — Problem 138
# Running this rebinds `solve` to the reference answer. Re-run your
# own cell before dd_check() again, or you are checking this one.
import torch as t

def solve(z, p):
    z[:] = z[p]
    return z


example = t.arange(6).reshape(3, 2)
print(solve(example, t.tensor([2, 0, 1])))


<!-- dd:dd-seg-numpy-inplace-out-2 -->

### chaining underscores for zero allocations


Because each underscore method returns the buffer it just wrote, they chain:
`b.add_(a)` computes a+b and stores it in b — zero new tensors. A run of these
(`a.div_(2)`, `a.neg_()`, …) evaluates a multi-step formula entirely inside
the input buffers.

Two disciplines make such code correct: track *what each buffer now holds*
(after `b.add_(a)`, the NAME b no longer means the original b), and order by
DATAFLOW — a value must be captured before the buffer holding its ingredient
is overwritten.


In [ ]:
import torch as t

# ((a + b) * (-a / 2)) in place only. Track buffer contents per step:
a = t.tensor([1.0, 2.0])
b = t.tensor([3.0, 4.0])
b.add_(a)                        # b now holds a + b
a.div_(2)                        # a now holds a / 2
a.neg_()                         # a now holds -a/2
a.mul_(b)                        # a now holds (a+b) * (-a/2)
assert a.tolist() == [-2.0, -6.0]
print("b holds a+b:", b)
print("a holds the final product:", a)




Why: order matters — b must absorb (a+b) BEFORE a is halved, since the
addition needs the original a. Reordering the same four calls breaks the
result; dataflow, not formula layout, dictates sequence.


<!-- dd:dd-q59 -->

### Problem 59 · faded — your turn

((a + b) * (-a / 2)) with ZERO new allocations — every step lands in a's or
b's buffer.

**Expected output** — run the cell below once `solve` is right and it should print this.

```text
tensor([-1.5000, -1.5000, -1.5000])
```


In [ ]:
import torch as t

def solve(a, b):
    """Compute ((a + b) * (-a / 2)) using only the two given buffers."""
    b._____(_____)
    a._____(2)
    a._____()
    a._____(b)
    return a


# Example run — the grader calls solve() with several array pairs.
example_a = t.ones(3)
example_b = t.ones(3) * 2
print(solve(example_a, example_b))


In [ ]:
# Did it work? Run this. (NameError → run the checker cell at
# the top of the notebook first: Runtime ▸ Run before.)
dd_check(59)


In [ ]:
#@title 💡 Solution — Problem 59
# Running this rebinds `solve` to the reference answer. Re-run your
# own cell before dd_check() again, or you are checking this one.
import torch as t
def solve(a, b):
    b.add_(a)
    a.div_(2)
    a.neg_()
    a.mul_(b)
    return a


example_a = t.ones(3)
example_b = t.ones(3) * 2
print(solve(example_a, example_b))


<!-- dd:dd-q527 -->

### Problem 527 · guided

Write a function solve(x, lo, hi) that clamps every entry of the float tensor x into the range [lo, hi] IN PLACE and returns that same tensor. The caller's own tensor object must change — building a clamped copy and returning it is a different contract and fails here. PyTorch marks the in-place version of an operation with a trailing underscore, and clamping has one.

**Expected output** — run the cell below once `solve` is right and it should print this.

```text
tensor([0., 2., 5.])
```


<details>
<summary>Hints</summary>

1. Read the contract first: the caller's own object has to change. That rules
   out every spelling that builds a clamped tensor and hands it back.
2. Clamping has a trailing-underscore form, `clamp_(lo, hi)`, and like every
   underscore method it writes into the receiver's buffer.
3. `x.clamp_(lo, hi)` and then `return x` — the call already returns that
   same tensor, so returning `x` is just saying so plainly.

</details>


In [ ]:
import torch
import torch as t

def solve(x, lo, hi):
    """Clamp x into [lo, hi] inside its own buffer, and return x."""
    return None


# Example run — the grader calls solve() with several different inputs,
# including edge cases. Your function must work for all of them.
example = (t.tensor([-1.0, 2.0, 7.0]), 0.0, 5.0)
print(solve(*example))


In [ ]:
# Did it work? Run this. (NameError → run the checker cell at
# the top of the notebook first: Runtime ▸ Run before.)
dd_check(527)


In [ ]:
#@title 💡 Solution — Problem 527
# Running this rebinds `solve` to the reference answer. Re-run your
# own cell before dd_check() again, or you are checking this one.
import torch
import torch as t

def solve(x, lo, hi):
    """Clamp x into [lo, hi] inside its own buffer, and return x."""
    x.clamp_(lo, hi)
    return x


example = (t.tensor([-1.0, 2.0, 7.0]), 0.0, 5.0)
print(solve(*example))


#### Common mistakes

- **"`z = z[p]` modifies z in place."** — It rebinds the NAME; the original
  buffer (and every other reference to it) is unchanged. In-place is
  `z[:] = z[p]` — assignment through the full slice.
- **"`x.sort()` sorts x."** — It does in NumPy; in PyTorch it returns a
  (values, indices) pair and leaves x untouched. There is no `sort_`, so an
  in-place sort is `x.copy_(x.sort().values)`.
- **"The underscore is a naming convention."** — It's a hard contract: the
  result is written into that exact buffer. Aliasing consequences included —
  `b.add_(a)` destroys the old b for all readers. Powerful, deliberate, and
  exactly what no-allocation drills demand.


<!-- dd:dd-kp-numpy-sliding-windows -->

## Sliding windows and moving averages

`numpy.sliding-windows`


"Every contiguous window of length w" — the substrate of moving averages,
local maxima, and convolutions — has one canonical constructor:

> **`x.unfold(dim, size, step)`**

For a length-n vector, `x.unfold(0, w, 1)` returns a **(n − w + 1, w) matrix
whose row i is `x[i : i + w]`** — every window, materialized as rows, WITHOUT
copying (it's a strided view into the original buffer; treat it as
read-only). Window count n − w + 1: one start position per element that still
has w−1 successors.

`unfold` takes the step as its third argument, so strided windows need no
extra slicing: `x.unfold(0, w, step)` is "windows starting every `step`
elements".

Once windows are rows, **window statistics are just dim-1 reductions**:

```python no-run
x.unfold(0, w, 1).mean(dim=1)    # moving average
x.unfold(0, w, 1).amax(dim=1)    # moving maximum
```

One alternative spelling earns its place:

- **cumsum trick for moving SUMS/averages**: a window sum is a difference of
  two running totals — `c[i+w] - c[i]` where c is the cumulative sum. O(n)
  with no (n, w) intermediate, the memory-friendly route when w is large.

Recognize the family by the phrase "every window / moving / rolling"; choose
the spelling by what's reduced (any statistic → unfold; sum/mean at scale →
cumsum).


Task: materialize all windows of length 3; compute the moving average two
ways and confirm they agree.


In [ ]:
import torch as t

x = t.arange(7, dtype=t.float32)            # 0 1 2 3 4 5 6

# 1. All windows as rows: shape (7-3+1, 3) = (5, 3).
wins = x.unfold(0, 3, 1)
assert tuple(wins.shape) == (5, 3)
assert wins[0].tolist() == [0.0, 1.0, 2.0]
assert wins[-1].tolist() == [4.0, 5.0, 6.0]

# 2a. Moving average = windows reduced along dim 1.
ma1 = wins.mean(dim=1)
assert ma1.tolist() == [1.0, 2.0, 3.0, 4.0, 5.0]

# 2b. cumsum spelling: window sum = difference of running totals.
c = t.cumsum(t.cat([t.zeros(1), x]), dim=0)   # c[i] = sum of first i elements
ma2 = (c[3:] - c[:-3]) / 3.0
assert t.allclose(ma2, ma1)
print("windows", tuple(wins.shape))
print(wins)
print("unfold+mean", ma1)
print("cumsum diff", ma2)




Why each step:

1. Checking the first and last rows of the window view fixes the boundary
   convention: the LAST window starts at n−w, so nothing hangs off the end —
   that's where the n−w+1 count comes from.
2. In the cumsum spelling, prepending a 0 makes the algebra uniform
   (`c[i+w] − c[i]` for every i, including i=0) — the standard trick for
   prefix-sum arithmetic. Note `t.cumsum` requires an explicit `dim`.
3. Two spellings, one answer: drills accept either; YOUR choice should follow
   constraints — arbitrary statistics need the window view, big-w sums want
   cumsum.


<!-- dd:dd-q117 -->

### Problem 117 · faded — your turn

The (n−w+1, w) matrix of all length-w windows.

**Expected output** — run the cell below once `solve` is right and it should print this.

```text
tensor([[0, 1, 2],
        [1, 2, 3],
        [2, 3, 4],
        [3, 4, 5],
        [4, 5, 6]])
```


In [ ]:
import torch as t

def solve(x, w):
    """Row i = x[i : i + w], every contiguous window."""
    return x._____(0, _____, 1)


# Example run — the grader calls solve() with several inputs.
print(solve(t.arange(7), 3))


In [ ]:
# Did it work? Run this. (NameError → run the checker cell at
# the top of the notebook first: Runtime ▸ Run before.)
dd_check(117)


In [ ]:
#@title 💡 Solution — Problem 117
# Running this rebinds `solve` to the reference answer. Re-run your
# own cell before dd_check() again, or you are checking this one.
import torch as t
def solve(x, w):
    return x.unfold(0, w, 1)


print(solve(t.arange(7), 3))


<!-- dd:dd-q166 -->

### Problem 166 · guided

Write a function solve(a, n) that takes a 1-D PyTorch tensor and a window size n (1 <= n <= len(a)), and returns the moving average over every contiguous window of n entries — output entry i is the mean of a[i : i + n], giving len(a) - n + 1 values. (An earlier drill fixes the window at 3; here the width is a parameter — the cumsum trick makes any width one subtraction.)

**Expected output** — run the cell below once `solve` is right and it should print this.

```text
tensor([ 1.,  2.,  3.,  4.,  5.,  6.,  7.,  8.,  9., 10., 11., 12., 13., 14.,
        15., 16., 17., 18.], dtype=torch.float64)
```


<details>
<summary>Hints</summary>

1. Moving average over every window of n entries — the window view + mean
   works; the cumsum route avoids materializing windows.
2. cumsum: a window's sum is `c[i+n] − c[i]`. Handle the offset by prepending
   a zero, or slice-shift the cumsum directly.
3. Either implementation passes — write the one you can verify, check
   endpoints against a tiny example by hand.

</details>


In [ ]:
import torch as t

def solve(a, n):
    """Return the moving average of a with window size n."""
    return None


# Example run — the grader calls solve() with several inputs.
print(solve(t.arange(20), 3))


In [ ]:
# Did it work? Run this. (NameError → run the checker cell at
# the top of the notebook first: Runtime ▸ Run before.)
dd_check(166)


In [ ]:
#@title 💡 Solution — Problem 166
# Running this rebinds `solve` to the reference answer. Re-run your
# own cell before dd_check() again, or you are checking this one.
import torch as t
def solve(a, n):
    ret = t.cumsum(a, dim=0, dtype=t.float64)
    ret[n:] = ret[n:] - ret[:-n]
    return ret[n - 1:] / n


print(solve(t.arange(20), 3))


<!-- dd:dd-q99 -->

### Problem 99 · independent

Write a function solve(x) that takes a 1-D PyTorch float tensor with at least 3 entries and returns the moving average over every window of 3 consecutive entries: output entry i is the mean of x[i], x[i+1], x[i+2]. The result has length len(x) - 2.

**Expected output** — run the cell below once `solve` is right and it should print this.

```text
tensor([1., 2., 3., 4., 5., 6., 7., 8.])
```


In [ ]:
import torch as t

def solve(x):
    """Return the length-3 moving average of x."""
    return None


# Example run — the grader calls solve() with several arrays.
print(solve(t.arange(10, dtype=t.float32)))


In [ ]:
# Did it work? Run this. (NameError → run the checker cell at
# the top of the notebook first: Runtime ▸ Run before.)
dd_check(99)


In [ ]:
#@title 💡 Solution — Problem 99
# Running this rebinds `solve` to the reference answer. Re-run your
# own cell before dd_check() again, or you are checking this one.
import torch as t
def solve(x):
    return x.unfold(0, 3, 1).mean(dim=1)


print(solve(t.arange(10, dtype=t.float32)))


<!-- dd:dd-q175 -->

### Problem 175 · independent

Write a function solve(z, w, step) that takes a 1-D PyTorch tensor, a window length w, and a positive stride step. Return the 2-D tensor whose rows are the length-w windows of z starting at positions 0, step, 2*step, ... (only windows that fit completely). Row count is (len(z) - w) // step + 1. (An earlier drill slides by 1; the stride parameter is what's new.)

**Expected output** — run the cell below once `solve` is right and it should print this.

```text
tensor([[0, 1, 2],
        [2, 3, 4],
        [4, 5, 6]])
```


In [ ]:
import torch as t

def solve(z, w, step):
    """Return length-w windows of z taken every `step` positions."""
    return None


# Example run — the grader calls solve() with several inputs.
print(solve(t.arange(8), 3, 2))


In [ ]:
# Did it work? Run this. (NameError → run the checker cell at
# the top of the notebook first: Runtime ▸ Run before.)
dd_check(175)


In [ ]:
#@title 💡 Solution — Problem 175
# Running this rebinds `solve` to the reference answer. Re-run your
# own cell before dd_check() again, or you are checking this one.
import torch as t
def solve(z, w, step):
    return z.unfold(0, w, 1)[::step]


print(solve(t.arange(8), 3, 2))


#### Common mistakes

- **"Window tasks need an explicit Python loop over starts."** — `unfold`
  materializes every start position as a row in one call; reductions do the
  rest. The loop survives only in the O(n·w) mental model, not the code.
- **"unfold copies n·w elements."** — It's a strided VIEW: no copy,
  negligible memory. (Consequence: don't write into it; clone first if you
  must mutate.)
- **"Moving average output has length n."** — 'valid' windows only:
  n − w + 1. Padding to length n is a separate, explicit decision
  (`t.nn.functional.pad` first) — the drills here use the valid convention.
